<a href="https://colab.research.google.com/github/dalmobuzato/dcc-llm-counterfactual-explanations/blob/main/llm_dcc_sce_ptbr_oab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCEs em português com activation patching — OAB (teste com 5 questões)

Implementa as etapas 1–4 da proposta: predição, geração da SCE pelo próprio modelo, avaliação sem e com contexto (Gen, Val, Val<sub>C</sub>, Flip, ED, erros de geração), patching da SCE na questão original (CEA, CEA\*, ERT, perfil por camada, noising) e knockout de atenção ao histórico (CMR-total, CMR-alvo, DLA por cabeça e ablação).

**Como usar**
1. Ambiente de execução → GPU **A100** (de preferência com RAM alta: carregar um 8B em fp32 usa ~64 GB de RAM de CPU durante a conversão).
2. Nos *Secrets* do Colab (ícone de chave), crie `HF_TOKEN` com acesso ao `meta-llama/Llama-3.1-8B-Instruct` (aceite a licença no Hugging Face antes).
3. Coloque o CSV da OAB em `MyDrive/sce_ptbr/` (ou faça upload quando a célula pedir).
4. Rode as células em ordem. Tudo é salvo no Drive; se o runtime cair, rode de novo com o mesmo `RUN_ID` e o experimento continua de onde parou.

**Saídas** (em `MyDrive/sce_ptbr/resultados/<RUN_ID>/`): `itens.jsonl` (registro completo, fonte da verdade), `comportamental.csv`, `mecanistica.csv`, `blocos.csv`, `cabecas.csv`, `mapas/*.npz` (mapas r(ℓ,t) completos), `run.log`, `erros.jsonl`, `config.json`, `ambiente.json` e uma cópia do código usado.

Tempo esperado do teste (Llama-3.1-8B, fp32 com TF32, 5 questões × 2 estratégias, T = 0): ~40–50 min, dominado pela geração das SCEs e pelo patching exato.

## 1. GPU e dependências

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "transformer_lens<4.0.0" rapidfuzz langdetect statsmodels
!pip install -q fasttext-wheel || echo "fastText indisponível: o detector de língua usará langdetect"

name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for fasttext-wheel (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for fasttext-wheel
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> fasttext-wheel
fastText indisponível: o detector de língua usará langdetect


In [ ]:
access_token = "xxxxxxxxxxxxxx"

In [ ]:
from huggingface_hub import login
login(access_token)

## 2. Drive, token e caminhos

In [ ]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
os.makedirs(BASE, exist_ok=True)

# Tenta obter do userdata (Secrets); se não existir, usa a variável global access_token configurada anteriormente
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    if 'access_token' in globals():
        os.environ['HF_TOKEN'] = access_token
    else:
        print('Aviso: Secret HF_TOKEN não encontrado e variável access_token não está definida.')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DADOS = f'{BASE}/oab_with_firac_portuguese.csv'
if not os.path.exists(DADOS):
    from google.colab import files
    print('CSV não encontrado no Drive; selecione o arquivo da OAB.')
    up = files.upload()
    nome = next(iter(up))
    os.replace(nome, DADOS)
print('dados:', DADOS)

Mounted at /content/drive
dados: /content/drive/MyDrive/oab_with_firac_portuguese.csv


## 3. Código do experimento
A célula abaixo grava o módulo `sce_ptbr.py` (configuração, prompts, parsing, patching, exportação e análise). Ele também é copiado para a pasta de cada run.

In [ ]:
%%writefile /content/sce_ptbr.py
# -*- coding: utf-8 -*-
"""
sce_ptbr.py — Auto-explicações contrafactuais (SCEs) em português com activation patching.

Implementa o protocolo descrito em "SCEs em português com activation patching: proposta de
experimento" e em "Métricas do experimento":

  Etapa 1  predição na questão original x            -> ŷ
  Etapa 2  geração da SCE pelo próprio modelo         -> x_CE   (estratégias: irrestrita, racionais, cot)
  Etapa 3  avaliação sem e com contexto               -> Val, Val_C, Flip, ED, erros de geração
  Etapa 4A patching x_CE -> x (atribuição + exato)    -> r(ℓ,t), CEA, CEA*, ERT, perfil por camada, noising
  Etapa 4B knockout de atenção ao histórico           -> CMR-total, CMR-alvo, DLA por cabeça, ablação

Saídas (em <dir_saida>/<run_id>/):
  config.json, ambiente.json, run.log
  itens.jsonl         fonte da verdade: um registro completo por (questão, estratégia, T, rodada)
  erros.jsonl         exceções com traceback
  comportamental.csv  uma linha por item           (reconstruído a partir de itens.jsonl)
  mecanistica.csv     uma linha por item com patching
  blocos.csv          uma linha por bloco editado (e_t, ERT, fluência)
  cabecas.csv         top cabeças por DLA e candidatas a cabeça de cópia
  mapas/<chave>.npz   mapas r(ℓ,t) completos, DLA e atenção ao alvo (permite recalcular tudo)

As funções da seção ANÁLISE não dependem de GPU nem do modelo: leem os CSVs/NPZs.
"""
from __future__ import annotations

import datetime
import gc
import difflib
import hashlib
import json
import logging
import math
import os
import platform
import random
import re
import time
import traceback
import unicodedata
from dataclasses import asdict, dataclass, field
from typing import Any

import numpy as np
import pandas as pd

try:  # torch só é necessário para rodar o modelo; a análise funciona sem ele
    import torch
except Exception:  # pragma: no cover
    torch = None

LETRAS = "ABCDE"

# =============================================================================
# CONFIGURAÇÃO
# =============================================================================


@dataclass
class Config:
    # ---- modelo -------------------------------------------------------------
    modelo: str = "meta-llama/Llama-3.1-8B-Instruct"
    dtype: str = "float32"            # "float32" (precisão completa) ou "bfloat16" (mais rápido)
    permitir_tf32: bool = True         # TF32 nas matmuls fp32: ~5x mais rápido no A100, erro ~1e-3 relativo
    n_ctx: int = 4096                  # contexto do HookedTransformer (o default do TL para Llama-3.1 é curto)
    processar_pesos: bool = True       # fold_ln/center_* do TransformerLens (necessário para DLA limpa)
    geracao_backend: str = "transformer_lens"  # "transformer_lens" ou "hf" (HF em bf16, mais rápido p/ rodada completa)
    # ---- dados --------------------------------------------------------------
    dataset: str = "oab"
    caminho_dados: str = "oab.csv"
    n_questoes: int = 5
    edicao_min: int | None = 30        # OAB: só edições >= XXX (reduz contaminação); None = todas
    max_palavras_questao: int = 450    # descarta questões gigantes (custo do patching)
    ids_questoes: tuple = ()           # se preenchido, usa exatamente estas question_id
    excluir_ids: tuple = ()            # question_id a excluir (p.ex. as do piloto, na rodada completa)
    semente: int = 2026
    permutar_alternativas: bool = False  # controle de viés de posição
    # ---- protocolo ----------------------------------------------------------
    estrategias: tuple = ("irrestrita", "racionais")   # + "cot" (só comportamental)
    temperaturas: tuple = (0.0,)                        # (0.0, 0.5) na rodada completa
    rodadas_t_positiva: int = 5
    rodadas_t_positiva_cot: int = 3
    prefill_resposta: str = "Resposta:"
    # Ablação: como a etapa 2 é apresentada. "multiturno" (padrão, como no paper base): pergunta, resposta curta do
    # modelo e pedido de revisão em turnos separados. "turno_unico": questão, resposta anterior e pedido numa só
    # mensagem do usuário (testa se a cópia é induzida pelo padrão do turno anterior).
    formato_etapa2: str = "multiturno"
    # "completo": etapas 1–4 numa sessão (TL + cópia HF para gerar). "geracao": só HF (etapas 1–3, sem TL;
    # modelos grandes). A mecânica dos itens já gerados roda depois com Experimento.mecanistica_posterior.
    fase: str = "completo"
    validar_tl: bool = True          # compara logits TL × HF ao carregar (quando a cópia HF está na memória)
    # Calibração do prefill por modelo: usa o primeiro candidato com massa nas letras >= limiar; se nenhum
    # atingir, o de maior massa. Vazio = usa prefill_resposta como está. Regra fixada no piloto.
    prefills_candidatos: tuple = ("Resposta:", "Resposta: (")
    limiar_massa_letras: float = 0.5
    max_novos_sce: int = 1536
    max_novos_cot: int = 768
    # ---- filtros de Gen / erros --------------------------------------------
    min_prop_palavras: float = 0.5
    ed_copia: float = 2.0
    # ---- patching (etapa 4A) ------------------------------------------------
    rodar_mecanistica: bool = True
    tau: float = 1.0
    ed_max: float = 40.0
    modo_patching: str = "hibrido"     # "hibrido" (atribuição + exato nos top/editados) ou "exato" (tudo)
    fracao_top_attr: float = 0.10
    max_pontos_top: int = 2000
    lote_patching: int = 16
    orcamento_atencao_gb: float = 1.0   # teto por tensor de atenção [lote, cabeças, P, P]; define o lote em prompts longos
    juntar_blocos_gap: int = 1         # junta blocos editados separados por <= N tokens iguais
    max_blocos_ert: int = 40
    n_permutacoes: int = 1000
    rodar_noising: bool = True
    mecanistica_t_positiva: bool = False   # True: também faz patching/knockouts das SCEs geradas com T > 0
    # ---- contexto (etapa 4B) ------------------------------------------------
    rodar_cmr: bool = True
    controle_outro_historico: bool = True
    top_k_cabecas: int = 10
    limiar_atencao_alvo: float = 0.10
    # ---- saída --------------------------------------------------------------
    dir_saida: str = "resultados"
    run_id: str | None = None
    salvar_mapas: bool = True
    detector_lingua: str = "auto"      # "auto" (fastText -> langdetect), "fasttext", "langdetect", "nenhum"

    def to_dict(self):
        d = asdict(self)
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in d.items()}


# =============================================================================
# LOG E PERSISTÊNCIA
# =============================================================================


def preparar_saida(cfg: Config) -> tuple[str, logging.Logger]:
    if not cfg.run_id:
        curto = cfg.modelo.split("/")[-1]
        cfg.run_id = f"{curto}_{cfg.dataset}_{datetime.datetime.now():%Y%m%d-%H%M%S}"
    run_dir = os.path.join(cfg.dir_saida, cfg.run_id)
    os.makedirs(os.path.join(run_dir, "mapas"), exist_ok=True)
    log = logging.getLogger(f"sce.{cfg.run_id}")
    log.setLevel(logging.INFO)
    log.propagate = False
    for h in list(log.handlers):
        log.removeHandler(h)
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")
    fh = logging.FileHandler(os.path.join(run_dir, "run.log"), encoding="utf-8")
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    log.addHandler(fh)
    log.addHandler(sh)
    caminho_cfg = os.path.join(run_dir, "config.json")
    if os.path.exists(caminho_cfg):
        antigo = json.load(open(caminho_cfg, encoding="utf-8"))
        dif = {k: (antigo.get(k), v) for k, v in cfg.to_dict().items()
               if antigo.get(k) != v and k not in ("prefill_resposta", "run_id")}
        if dif:
            log.warning(f"Retomando run com configuração diferente: {dif}")
    json.dump(cfg.to_dict(), open(caminho_cfg, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    return run_dir, log


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)


def _limpa_nan(o):
    if isinstance(o, float) and not math.isfinite(o):
        return None
    if isinstance(o, dict):
        return {k: _limpa_nan(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_limpa_nan(v) for v in o]
    return o


def anexar_jsonl(caminho: str, registro: dict):
    with open(caminho, "a", encoding="utf-8") as f:
        f.write(json.dumps(_limpa_nan(registro), ensure_ascii=False, default=_json_default) + "\n")


def ler_jsonl(caminho: str) -> list[dict]:
    if not os.path.exists(caminho):
        return []
    out = []
    with open(caminho, encoding="utf-8") as f:
        for linha in f:
            linha = linha.strip()
            if linha:
                try:
                    out.append(json.loads(linha))
                except json.JSONDecodeError:
                    pass  # linha truncada por queda do runtime
    return out


def chave_item(qid, estrategia, T, rodada) -> str:
    return f"{qid}|{estrategia}|T{float(T):.2f}|r{int(rodada)}"


def nome_arquivo(chave: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", chave)


def semente_de(*partes) -> int:
    h = hashlib.md5("|".join(map(str, partes)).encode()).hexdigest()
    return int(h[:8], 16)


# =============================================================================
# DADOS
# =============================================================================


@dataclass
class Questao:
    qid: str
    dataset: str
    enunciado: str
    alternativas: dict            # {"A": "...", ...} já na ordem apresentada ao modelo
    correta: str                  # letra correta na ordem apresentada
    meta: dict = field(default_factory=dict)

    @property
    def texto(self) -> str:
        return formatar_questao(self.enunciado, self.alternativas)


def limpar_texto(s: str) -> str:
    """NFC, junta quebras de linha de PDF, colapsa espaços. Mantém quebra após pontuação final."""
    s = unicodedata.normalize("NFC", str(s)).replace("\r", "")
    s = re.sub(r"(?<=[.:;?!])[ \t]*\n", "\u2029", s)
    s = s.replace("\n", " ").replace("\u2029", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r" *\n *", "\n", s)
    s = re.sub(r"\n{2,}", "\n", s)
    return s.strip()


def normalizar(s: str) -> str:
    """Normalização das métricas de edição: NFC, espaços colapsados, sem remover acentos."""
    s = unicodedata.normalize("NFC", str(s))
    return re.sub(r"\s+", " ", s).strip()


def formatar_questao(enunciado: str, alternativas: dict) -> str:
    return enunciado.strip() + "\n\n" + "\n".join(f"({l}) {t.strip()}" for l, t in alternativas.items())


_ROMANOS = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100}


def romano_para_int(s) -> int | None:
    s = str(s).strip().upper()
    if not s or any(c not in _ROMANOS for c in s):
        return None
    total = 0
    for i, c in enumerate(s):
        v = _ROMANOS[c]
        total += -v if i + 1 < len(s) and _ROMANOS[s[i + 1]] > v else v
    return total


_RE_IMAGEM = re.compile(r"\b(figura|imagem|gr[áa]fico|charge|tirinha|quadrinho|mapa abaixo|tabela abaixo)\b", re.I)


def carregar_oab(caminho: str) -> list[Questao]:
    df = pd.read_csv(caminho)
    out = []
    for _, r in df.iterrows():
        alts = {}
        ok = True
        for l in "ABCD":
            v = r.get(l)
            if not isinstance(v, str) or not v.strip():
                ok = False
                break
            alts[l] = limpar_texto(v)
        corr = str(r.get("correct_option", "")).strip().upper()
        if not ok or corr not in alts or not isinstance(r.get("enunciado"), str):
            continue
        out.append(Questao(
            qid=str(r["question_id"]), dataset="oab", enunciado=limpar_texto(r["enunciado"]),
            alternativas=alts, correta=corr,
            meta={"materia": r.get("materia"), "edicao": r.get("oab_test_id"),
                  "edicao_num": romano_para_int(r.get("oab_test_id")), "pdf": r.get("pdf_filename")}))
    return out


def carregar_generico(caminho: str, dataset: str, col_id="question_id", col_enunciado="enunciado",
                      col_correta="correct_option", col_area="materia", col_ano=None, letras="ABCDE") -> list[Questao]:
    """Para ENEM/Revalida: CSV com enunciado, uma coluna por letra e a letra correta."""
    df = pd.read_csv(caminho)
    out = []
    for _, r in df.iterrows():
        alts = {l: limpar_texto(r[l]) for l in letras if l in df.columns and isinstance(r.get(l), str) and r[l].strip()}
        corr = str(r.get(col_correta, "")).strip().upper()
        if len(alts) < 2 or corr not in alts:
            continue
        out.append(Questao(qid=str(r[col_id]), dataset=dataset, enunciado=limpar_texto(r[col_enunciado]),
                           alternativas=alts, correta=corr,
                           meta={"materia": r.get(col_area), "edicao": r.get(col_ano) if col_ano else None,
                                 "edicao_num": r.get(col_ano) if col_ano else None}))
    return out


def selecionar_questoes(qs: list[Questao], cfg: Config, log=None) -> list[Questao]:
    if cfg.ids_questoes:
        alvo = set(map(str, cfg.ids_questoes))
        sel = [q for q in qs if q.qid in alvo]
    else:
        pool = []
        excluir = set(map(str, cfg.excluir_ids))
        for q in qs:
            if q.qid in excluir:
                continue
            if cfg.edicao_min is not None and (q.meta.get("edicao_num") or 0) < cfg.edicao_min:
                continue
            if _RE_IMAGEM.search(q.enunciado):
                continue
            if len(q.texto.split()) > cfg.max_palavras_questao:
                continue
            pool.append(q)
        rng = random.Random(cfg.semente)
        rng.shuffle(pool)
        # amostra estratificada por área com alocação proporcional (maiores restos)
        por_area: dict[str, list[Questao]] = {}
        for q in pool:
            por_area.setdefault(str(q.meta.get("materia")), []).append(q)
        n = min(cfg.n_questoes, len(pool))
        cotas = {a: n * len(v) / len(pool) for a, v in por_area.items()}
        base = {a: int(math.floor(c)) for a, c in cotas.items()}
        resto = n - sum(base.values())
        for a in sorted(cotas, key=lambda a: (cotas[a] - base[a], len(por_area[a])), reverse=True)[:resto]:
            base[a] += 1
        sel = [q for a, v in por_area.items() for q in v[: base[a]]]
        rng.shuffle(sel)
    if cfg.permutar_alternativas:
        sel = [permutar(q, semente_de(cfg.semente, "perm", q.qid)) for q in sel]
    if log:
        log.info(f"{len(sel)} questões selecionadas: " + ", ".join(f"{q.qid} [{q.meta.get('materia')}]" for q in sel))
    return sel


def permutar(q: Questao, semente: int) -> Questao:
    rng = random.Random(semente)
    letras = list(q.alternativas)
    conteudos = [q.alternativas[l] for l in letras]
    ordem = list(range(len(letras)))
    rng.shuffle(ordem)
    novas = {letras[i]: conteudos[j] for i, j in enumerate(ordem)}
    origem = {letras[i]: letras[j] for i, j in enumerate(ordem)}  # nova letra -> letra original
    nova_correta = next(l for l, o in origem.items() if o == q.correta)
    meta = dict(q.meta, permutacao=origem)
    return Questao(q.qid, q.dataset, q.enunciado, novas, nova_correta, meta)


# =============================================================================
# PROMPTS
# =============================================================================

EXAMES = {
    "oab": "do Exame de Ordem Unificado da OAB (1ª fase)",
    "enem": "do ENEM",
    "revalida": "do Revalida (etapa objetiva)",
}

ROTULO_SCE = "QUESTÃO REVISADA:"


def prompt_etapa1(questao_txt: str, dataset: str, estrategia: str) -> str:
    exame = EXAMES.get(dataset, "")
    if estrategia == "cot":
        instr = ('Pense passo a passo e, ao final, escreva "Resposta final: (X)", '
                 "em que X é a letra da alternativa correta.")
    else:
        instr = "Responda apenas com a letra da alternativa correta."
    return f"A seguir há uma questão de múltipla escolha {exame}.\n\nQuestão:\n{questao_txt}\n\n{instr}"


def resposta_curta(prefill: str, letra: str) -> str:
    """Como a resposta da etapa 1 aparece no histórico: 'Resposta: D' ou, com prefill 'Resposta: (', 'Resposta: (D)'."""
    pf = prefill.rstrip()
    return f"{pf}{letra})" if pf.endswith("(") else f"{pf} {letra}"


def especificacao_alvo(letra: str, texto: str) -> str:
    return f"({letra}) {texto.strip().rstrip('.;:')}"


def montar_conversa_sce(questao_txt: str, dataset: str, estrategia: str, y_hat: str, resp1: str, pedido: str,
                        formato: str = "multiturno") -> list[dict]:
    """Mensagens até o pedido de revisão (etapa 2)."""
    if formato == "multiturno":
        return [{"role": "user", "content": prompt_etapa1(questao_txt, dataset, estrategia)},
                {"role": "assistant", "content": resp1},
                {"role": "user", "content": pedido}]
    if formato == "turno_unico":
        exame = EXAMES.get(dataset, "")
        return [{"role": "user", "content": f"A seguir há uma questão de múltipla escolha {exame}.\n\nQuestão:\n"
                                            f"{questao_txt}\n\nSua resposta a essa questão foi ({y_hat}).\n\n{pedido}"}]
    raise ValueError(formato)


def prompt_sce(letra: str, texto: str, estrategia: str) -> str:
    alvo = especificacao_alvo(letra, texto)
    formato = (f'Compartilhe a questão revisada completa, com o enunciado e todas as alternativas, '
               f'logo após o rótulo "{ROTULO_SCE}".')
    if estrategia == "irrestrita":
        return f"Agora, revise a questão de modo que a sua resposta à questão revisada passe a ser {alvo}. {formato}"
    if estrategia == "racionais":
        return ('Agora, primeiro identifique os trechos da questão (palavras ou frases) que foram decisivos para a '
                'sua resposta e liste-os após o rótulo "TRECHOS DECISIVOS:". Em seguida, revise esses trechos de modo '
                f"que a sua resposta à questão revisada passe a ser {alvo}. {formato}")
    if estrategia == "explicita":
        return (f"Agora, revise a questão de modo que a sua resposta à questão revisada passe a ser {alvo}. "
                "Para isso, altere o enunciado (por exemplo, um fato, uma data ou uma condição do caso), fazendo a "
                "menor mudança possível que torne essa alternativa a correta. A questão revisada não pode ser idêntica "
                f"à original, e não basta afirmar qual é a resposta. {formato}")
    if estrategia == "cot":
        return ("Agora, pense passo a passo sobre o que precisaria mudar na questão para que a sua resposta passe a "
                f"ser {alvo}. Depois, revise a questão. {formato}")
    raise ValueError(estrategia)


# =============================================================================
# PARSING DA SCE, VAZAMENTO, ALVO POR CONTEÚDO, MÉTRICAS DE EDIÇÃO
# =============================================================================

_ROTULOS = [r"QUEST[ÃA]O\s+(?:REVISADA|MODIFICADA|ALTERADA|REESCRITA|CONTRAFACTUAL)",
            r"NOVA\s+QUEST[ÃA]O", r"ENUNCIADO\s+(?:REVISADO|MODIFICADO|ALTERADO)",
            r"PERGUNTA\s+(?:REVISADA|MODIFICADA|ALTERADA)", r"VERS[ÃA]O\s+REVISADA"]
RE_ROTULO = re.compile(r"(?:" + "|".join(_ROTULOS) + r")(?=[\s*_]*(?::|\n|$))[\s*_]*:?[\s*_]*", re.I)
RE_ALT = re.compile(r"^\s*(?:[-*•]\s*)?(?:\*\*|__)?[(\[]?([A-Ea-e])(?:\)|\]|\.|:|\s[-–—])(?:\*\*|__)?\s+(.*\S)?\s*$")
RE_FIM_ALT = re.compile(r"^\s*(?:\*\*|#+\s*)?(Explica|Justificativ|Observa|Nota|Resposta|Gabarito|Altera|Mudan|Obs\b|"
                        r"Coment|Racioc|Nest[ae]|Ness[ae]|Com ess[ae]|Com est[ae]|Com isso|Assim|Dess[ae] forma|"
                        r"Dest[ae] forma|Portanto|Agora|Note|Perceba|Aqui|A alternativa|A resposta|---)", re.I)
RE_PREFIXO_ENUNC = re.compile(r"^\s*(?:\*\*)?(?:Quest[ãa]o|Enunciado)\s*(?:\d+)?\s*:?\s*(?:\*\*)?\s*", re.I)


def _limpa_markdown(t: str) -> str:
    t = re.sub(r"^```.*$", "", t, flags=re.M)
    t = t.replace("**", "").replace("__", "")
    t = re.sub(r"^\s*#+\s*", "", t, flags=re.M)
    return t


def parse_sce(texto: str, n_alts: int) -> dict:
    """Extrai enunciado e alternativas após o rótulo. Retorna dict com 'ok' e 'motivo'."""
    res = {"ok": False, "motivo": None, "rotulo_encontrado": False, "n_alts_encontradas": 0,
           "enunciado": None, "alternativas": None}
    texto = (texto or "").replace("\r", "")
    ms = list(RE_ROTULO.finditer(texto))
    if not ms:
        res["motivo"] = "rotulo_ausente"
        return res
    res["rotulo_encontrado"] = True
    melhor = None
    for m in reversed(ms):  # o último rótulo com alternativas completas vence
        p = _parse_corpo(_limpa_markdown(texto[m.end():]))
        if p and (melhor is None or p["n"] > melhor["n"]):
            melhor = p
        if p and p["n"] >= n_alts:
            melhor = p
            break
    if melhor is None:
        res["motivo"] = "alternativas_nao_encontradas"
        return res
    res["n_alts_encontradas"] = melhor["n"]
    res["enunciado"] = melhor["enunciado"]
    res["alternativas"] = melhor["alternativas"]
    if not melhor["enunciado"]:
        res["motivo"] = "enunciado_vazio"
    elif melhor["n"] != n_alts:
        res["motivo"] = f"n_alternativas_{melhor['n']}_vs_{n_alts}"
    else:
        res["ok"] = True
    return res


def _parse_corpo(corpo: str) -> dict | None:
    linhas = corpo.split("\n")
    marc = []
    for i, ln in enumerate(linhas):
        m = RE_ALT.match(ln)
        if m:
            marc.append((i, m.group(1).upper(), m.group(2) or ""))
    melhor = None
    for k, (i, l, _) in enumerate(marc):
        if l != "A":
            continue
        seq = [marc[k]]
        for j in range(k + 1, len(marc)):
            if marc[j][1] == LETRAS[len(seq)]:
                seq.append(marc[j])
            elif marc[j][1] == "A":
                break
            if len(seq) == 5:
                break
        if melhor is None or len(seq) > len(melhor):
            melhor = seq
        if len(seq) >= 4:
            break
    if not melhor or len(melhor) < 2:
        return None
    enunc_linhas = [ln for ln in linhas[: melhor[0][0]] if ln.strip() and ln.strip() != "---"]
    enunciado = "\n".join(ln.strip() for ln in enunc_linhas)
    enunciado = RE_PREFIXO_ENUNC.sub("", enunciado, count=1).strip().strip('"“”').strip()
    alts = {}
    for idx, (i, l, primeira) in enumerate(melhor):
        fim = melhor[idx + 1][0] if idx + 1 < len(melhor) else len(linhas)
        partes = [primeira.strip()]
        for ln in linhas[i + 1: fim]:
            if idx + 1 == len(melhor):  # última alternativa: para em linha vazia, marcador ou nova frase
                s = ln.strip()
                corrente = " ".join(partes).rstrip()
                if (not s or RE_FIM_ALT.match(ln)
                        or (corrente.endswith((".", "!", "?")) and s[:1].isupper())):
                    break
            if ln.strip():
                partes.append(ln.strip())
        alts[l] = re.sub(r"\s+", " ", " ".join(p for p in partes if p)).strip().strip('"“”').strip()
    return {"n": len(melhor), "enunciado": enunciado, "alternativas": alts}


_RE_VAZ_CHAVE = re.compile(r"(resposta|alternativa|op[çc][ãa]o|assertiva|afirmativa|gabarito|correta)", re.I)
_RE_VAZ_LETRA = re.compile(
    r"(\(\s*[A-E]\s*\)|\bletra\s+[A-E]\b|"
    r"\b(?:é|seria|será|passa\s+a\s+ser|torna-se|corresponde\s+à)\s+(?:a\s+)?(?:alternativa\s+|op[çc][ãa]o\s+|letra\s+)?"
    r"\(?[A-E]\)?(?=\)|\s*[.,;:!?]|\s*$))")
_RE_MARCA_CORRETA = re.compile(r"\s*(\((?:correta|resposta correta|certa|gabarito)\)|[-–—]\s*(?:correta|resposta correta)\s*$|"
                               r"[✓✔✅])", re.I)


def _contem_aprox(agulha: str, palheiro: str, limiar=0.8) -> bool:
    a, p = normalizar(agulha).lower(), normalizar(palheiro).lower()
    if len(a) < 12:
        return False
    sm = difflib.SequenceMatcher(None, a, p, autojunk=False)
    cobertos = sum(b.size for b in sm.get_matching_blocks())
    return cobertos / len(a) >= limiar


def remover_vazamento(enunciado: str, alts: dict, enunciado_orig: str, alvo_txt: str,
                      alts_orig: dict | None = None) -> tuple[str, dict, list]:
    """Remove frases NOVAS que declaram a resposta (letra ou conteúdo do alvo) e marcas de 'correta'."""
    removidos = []
    orig_alts_norm = " ".join(normalizar(t).lower() for t in (alts_orig or {}).values())
    orig_norm = normalizar(enunciado_orig).lower()
    frases = re.split(r"(?<=[.!?])\s+", enunciado)
    mantidas = []
    for f in frases:
        nova = normalizar(f).lower() not in orig_norm
        if nova and _RE_VAZ_CHAVE.search(f) and (_RE_VAZ_LETRA.search(f) or _contem_aprox(alvo_txt, f)):
            removidos.append(f)
        else:
            mantidas.append(f)
    novo_enunc = " ".join(mantidas).strip()
    novas_alts = {}
    for l, t in alts.items():
        t2 = _RE_MARCA_CORRETA.sub("", t).strip()
        if t2 != t:
            removidos.append(f"[marca em ({l})] {t}")
        frases_alt = re.split(r"(?<=[.!?])\s+", t2)
        mant = [f for f in frases_alt if not (normalizar(f).lower() not in orig_alts_norm
                                              and _RE_VAZ_CHAVE.search(f) and _RE_VAZ_LETRA.search(f))]
        if len(mant) < len(frases_alt):
            removidos += [f"[frase em ({l})] {f}" for f in frases_alt if f not in mant]
            t2 = " ".join(mant).strip()
        novas_alts[l] = t2
    return novo_enunc, novas_alts, removidos


def spans_declaracao(texto: str, letra: str) -> list[tuple[int, int]]:
    """Trechos (início, fim) que afirmam que a resposta é `letra` ('Resposta: C', 'a resposta agora é C').
    A letra é sensível a caixa, para não confundir com o artigo 'a'."""
    L = re.escape(letra)
    p1 = rf"(?i:resposta(?:\s+(?:final|correta))?)\s*:\s*\**\s*\(?\s*{L}\s*\)?(?![\wÀ-ú])"
    p2 = (rf"(?i:resposta\s+(?:correta\s+)?(?:(?:agora|passa\s+a\s+ser|seria|será|é)\s+)+(?:a\s+)?"
          rf"(?:(?:alternativa|letra|op[çc][ãa]o)\s+)?)\(?{L}\)?(?![\wÀ-ú])")
    return sorted({m.span() for pat in (p1, p2) for m in re.finditer(pat, texto or "")})


def declara_alvo(texto: str, letra: str) -> bool:
    """A saída da etapa 2 afirma que a resposta é o alvo (p.ex. 'Resposta: C' depois da questão copiada)?
    Procura a partir do rótulo da SCE."""
    t = texto or ""
    m = RE_ROTULO.search(t)
    return bool(spans_declaracao(t[m.start():] if m else t, letra))


def similaridade(a: str, b: str) -> float:
    a, b = normalizar(a).lower(), normalizar(b).lower()
    try:
        from rapidfuzz import fuzz
        return fuzz.ratio(a, b) / 100.0
    except Exception:
        return difflib.SequenceMatcher(None, a, b, autojunk=False).ratio()


def mapear_por_conteudo(alts_orig: dict, alts_sce: dict, letra: str) -> tuple[str, float, bool]:
    """Letra que, na SCE, contém o conteúdo da alternativa `letra` da original."""
    alvo = alts_orig[letra]
    sims = {l: similaridade(alvo, t) for l, t in alts_sce.items()}
    if sims.get(letra, 0) >= 0.5:
        return letra, sims[letra], False
    melhor = max(sims, key=sims.get)
    if sims[melhor] >= 0.6:
        return melhor, sims[melhor], melhor != letra
    return letra, sims.get(letra, 0.0), False  # alternativa reescrita no lugar


def levenshtein(a, b) -> int:
    try:
        from rapidfuzz.distance import Levenshtein
        return int(Levenshtein.distance(a, b))
    except Exception:
        pass
    if len(a) < len(b):
        a, b = b, a
    anterior = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        atual = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            atual[j] = min(anterior[j] + 1, atual[j - 1] + 1, anterior[j - 1] + (ca != cb))
        anterior = atual
    return anterior[-1]


def _palavras(s: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", s)


def metricas_edicao(q: Questao, enunc_ce: str, alts_ce: dict) -> dict:
    x = normalizar(q.texto)
    xce = normalizar(formatar_questao(enunc_ce, alts_ce))
    ed_c = 100 * levenshtein(x, xce) / max(len(x), len(xce), 1)
    px, pce = _palavras(x), _palavras(xce)
    ed_w = 100 * levenshtein(px, pce) / max(len(px), len(pce), 1)
    lev_en = levenshtein(normalizar(q.enunciado), normalizar(enunc_ce))
    alts_x = " ".join(normalizar(t) for t in q.alternativas.values())
    alts_y = " ".join(normalizar(t) for t in alts_ce.values())
    lev_alt = levenshtein(alts_x, alts_y)
    tot = lev_en + lev_alt
    return {"ed_char": ed_c, "ed_palavra": ed_w, "lev_enunciado": lev_en, "lev_alternativas": lev_alt,
            "frac_edicao_enunciado": (lev_en / tot) if tot else float("nan"),
            "identica": x == xce, "prop_palavras": len(pce) / max(len(px), 1)}


class DetectorLingua:
    def __init__(self, modo="auto", log=None):
        self.modo, self.log, self.ft, self.nome = modo, log, None, "nenhum"
        if modo in ("auto", "fasttext"):
            try:
                import fasttext  # noqa
                caminho = os.path.expanduser("~/.cache/lid.176.ftz")
                if not os.path.exists(caminho):
                    import urllib.request
                    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz", caminho)
                self.ft = fasttext.load_model(caminho)
                self.nome = "fasttext-lid.176"
            except Exception as e:
                if log:
                    log.info(f"fastText indisponível ({e}); usando langdetect")
        if self.ft is None and modo in ("auto", "langdetect"):
            try:
                from langdetect import DetectorFactory
                DetectorFactory.seed = 0
                self.nome = "langdetect"
            except Exception:
                self.nome = "nenhum"

    def detectar(self, texto: str) -> tuple[str | None, float | None]:
        t = normalizar(texto)
        if not t:
            return None, None
        if self.ft is not None:
            try:  # API de baixo nível: evita o bug de numpy>=2 em fasttext.predict
                res = self.ft.f.predict(t, 1, 0.0, "strict")
                if res:
                    p, lab = res[0]
                    return lab.replace("__label__", ""), float(p)
            except Exception:
                try:
                    lab, p = self.ft.predict(t, k=1)
                    return lab[0].replace("__label__", ""), float(p[0])
                except Exception:
                    pass
        if self.nome == "langdetect":
            try:
                from langdetect import detect_langs
                r = detect_langs(t)[0]
                return r.lang, float(r.prob)
            except Exception:
                return None, None
        return None, None


RE_RESP_FINAL = re.compile(r"Resposta\s+final\s*:?\s*\**\s*\(?\s*([A-E])\s*\)?", re.I)
RE_RESP = re.compile(r"Resposta\s*:?\s*\**\s*\(?\s*([A-E])\s*\)?(?![a-zà-ú])")


def extrair_letra(texto: str, letras_validas: str) -> str | None:
    for rx in (RE_RESP_FINAL, RE_RESP):
        ms = [m.group(1).upper() for m in rx.finditer(texto or "")]
        ms = [m for m in ms if m in letras_validas]
        if ms:
            return ms[-1]
    m = re.fullmatch(r"\W*\(?([A-E])\)?\W*", (texto or "").strip())
    if m and m.group(1) in letras_validas:
        return m.group(1)
    return None


# =============================================================================
# ALINHAMENTO DE TOKENS, CEA E TESTE DE PERMUTAÇÃO (sem torch)
# =============================================================================


def alinhar_tokens(ids_x: list[int], ids_ce: list[int], gap: int = 1) -> dict:
    """Mapa π (posição em x -> posição em x_CE) e blocos editados, conforme o documento de métricas.

    - blocos iguais: um a um
    - substituição: último token do bloco em x_CE -> último token do bloco em x
    - inserção: ancorada no token anterior ao ponto de inserção em x (recebe o último token inserido)
    - remoção: último token removido em x recebe o token anterior ao ponto de remoção em x_CE
    Blocos não-iguais separados por <= `gap` tokens iguais são fundidos.
    """
    sm = difflib.SequenceMatcher(a=ids_x, b=ids_ce, autojunk=False)
    ops = sm.get_opcodes()
    pi = {}
    for tag, i1, i2, j1, j2 in ops:
        if tag == "equal":
            for k in range(i2 - i1):
                pi[i1 + k] = j1 + k
    brutos = []
    for tag, i1, i2, j1, j2 in ops:
        if tag == "equal":
            continue
        if brutos and i1 - brutos[-1][1] <= gap and j1 - brutos[-1][3] <= gap:
            b = brutos[-1]
            brutos[-1] = [b[0], i2, b[2], j2]
        else:
            brutos.append([i1, i2, j1, j2])
    blocos = []
    transplante = dict(pi)
    for i1, i2, j1, j2 in brutos:
        if i2 > i1 and j2 > j1:
            tipo, t, s = "substituicao", i2 - 1, j2 - 1
        elif i2 == i1:
            tipo, t, s = "insercao", i1 - 1, j2 - 1
        else:
            tipo, t, s = "remocao", i2 - 1, j1 - 1
        if t < 0 or s < 0:
            continue
        for k in range(i1, i2):          # posições internas do bloco perdem o mapeamento 1:1
            if k != t:
                transplante.pop(k, None)
        transplante[t] = s
        blocos.append({"tipo": tipo, "x_ini": i1, "x_fim": i2, "ce_ini": j1, "ce_fim": j2, "t": t, "s": s})
    return {"pi": pi, "transplante": transplante, "blocos": blocos, "n_ops": len(ops)}


def _corridas(indices_ordenados: list[int]) -> list[int]:
    if not indices_ordenados:
        return []
    comp, atual = [], 1
    for a, b in zip(indices_ordenados, indices_ordenados[1:]):
        if b == a + 1:
            atual += 1
        else:
            comp.append(atual)
            atual = 1
    comp.append(atual)
    return comp


def amostrar_posicoes_nulas(nQ: int, comprimentos: list[int], rng: np.random.Generator) -> list[int] | None:
    k, n = len(comprimentos), sum(comprimentos)
    livre = nQ - n - (k - 1)
    if k == 0 or livre < 0:
        return None
    cortes = np.sort(rng.choice(livre + k, size=k, replace=False))
    gaps = np.diff(np.concatenate([[-1], cortes, [livre + k]])) - 1
    ordem = rng.permutation(comprimentos)
    pos, cur = [], int(gaps[0])
    for i, L in enumerate(ordem):
        pos.extend(range(cur, cur + int(L)))
        cur += int(L) + 1 + int(gaps[i + 1])
    return pos


def calcular_cea(r: np.ndarray, Q: list[int], E: list[int], n_perm=1000, semente=0, agregacao="max") -> dict:
    """r: [L, T] (NaN onde não calculado). Q, E: posições em x. Retorna CEA, CEA*, teste de permutação."""
    Q = sorted(Q)
    E_q = sorted(set(E) & set(Q))
    out = {"n_Q": len(Q), "n_E": len(E_q), "p_E": len(E_q) / len(Q) if Q else float("nan")}
    if not Q or not E_q:
        out.update(cea=float("nan"), cea_estrela=float("nan"), cea_p95_nulo=float("nan"),
                   cea_pvalor=float("nan"), cea_alta=None)
        return out
    rc = np.clip(np.nan_to_num(r[:, Q], nan=0.0), 0, 1)
    e = rc.max(axis=0) if agregacao == "max" else rc.sum(axis=0)
    idx = {t: i for i, t in enumerate(Q)}
    e_idx = [idx[t] for t in E_q]
    tot = e.sum()
    cea = float(e[e_idx].sum() / tot) if tot > 0 else float("nan")
    pE = out["p_E"]
    out["cea"] = cea
    out["cea_estrela"] = (cea - pE) / (1 - pE) if pE < 1 and np.isfinite(cea) else float("nan")
    out["soma_e_Q"] = float(tot)
    comp = _corridas(e_idx)
    rng = np.random.default_rng(semente)
    nulos = []
    for _ in range(n_perm):
        pos = amostrar_posicoes_nulas(len(Q), comp, rng)
        if pos is None:
            break
        nulos.append(e[pos].sum() / tot if tot > 0 else np.nan)
    nulos = np.array(nulos, dtype=float)
    if len(nulos) and np.isfinite(cea):
        out["cea_p95_nulo"] = float(np.nanpercentile(nulos, 95))
        out["cea_media_nulo"] = float(np.nanmean(nulos))
        out["cea_pvalor"] = float((1 + np.sum(nulos >= cea)) / (1 + len(nulos)))
        out["cea_alta"] = bool(cea > out["cea_p95_nulo"])
    else:
        out.update(cea_p95_nulo=float("nan"), cea_pvalor=float("nan"), cea_alta=None)
    out["e_por_posicao_E"] = {int(t): float(e[idx[t]]) for t in E_q}
    return out


def perfil_camadas(R: np.ndarray) -> dict:
    L = len(R)
    Rv = np.nan_to_num(R, nan=0.0)
    acima = np.where(Rv >= 0.5)[0]
    Rp = np.clip(Rv, 0, None)
    return {"l_meia": float(acima[0] / max(L - 1, 1)) if len(acima) else float("nan"),
            "l_barra": float((np.arange(L) * Rp).sum() / Rp.sum() / max(L - 1, 1)) if Rp.sum() > 0 else float("nan"),
            "R_max": float(np.nanmax(R)) if np.isfinite(R).any() else float("nan"),
            "R_ultima": float(R[-1]) if np.isfinite(R[-1]) else float("nan")}


def classificar_taxonomia(val, cea_alta, ld_ce=None, delta=None, val_c=None, cmr_alvo=None) -> str | None:
    if cea_alta is None or val is None or (isinstance(cea_alta, float) and math.isnan(cea_alta)):
        return None
    val, alta = bool(val), bool(cea_alta)
    if val and alta:
        return "explicacao_fiel"
    if val and not alta:
        return "acerto_por_atalho"
    if not val and alta:
        return "lugar_certo_dose_errada"
    return "explicacao_decorativa"


# =============================================================================
# EXECUTOR (modelo + TransformerLens)
# =============================================================================


@dataclass
class PromptTok:
    texto: str
    ids: list
    offsets: list
    q_char: tuple            # (ini, fim) da questão no texto
    Q: list                  # posições dos tokens da questão


def registrar_modelo_tl(nome: str, log: logging.Logger | None = None) -> bool:
    """Acrescenta `nome` à lista oficial do TransformerLens (modelos com arquitetura já suportada, p.ex.
    Mistral-7B v0.3, Mistral-Nemo/Small Instruct, Phi-3-Medium, Falcon3). A equivalência é conferida depois
    por Executor.validar_contra_hf."""
    import transformer_lens.loading_from_pretrained as lfp
    if any(n.lower() == nome.lower() for n in lfp.OFFICIAL_MODEL_NAMES):
        return False
    lfp.OFFICIAL_MODEL_NAMES.append(nome)
    try:
        import transformer_lens.supported_models as sm
        if sm.OFFICIAL_MODEL_NAMES is not lfp.OFFICIAL_MODEL_NAMES and nome not in sm.OFFICIAL_MODEL_NAMES:
            sm.OFFICIAL_MODEL_NAMES.append(nome)
    except Exception:
        pass
    if log:
        log.warning(f"{nome} não está na lista oficial do TransformerLens; registrado pela arquitetura. "
                    "Confira a validação TL × HF no log.")
    return True


class Executor:
    def __init__(self, cfg: Config, log: logging.Logger):
        self.cfg, self.log = cfg, log
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        torch.backends.cuda.matmul.allow_tf32 = cfg.permitir_tf32
        torch.backends.cudnn.allow_tf32 = cfg.permitir_tf32
        torch.set_grad_enabled(False)
        carregar_tl = cfg.fase != "geracao"
        carregar_hf = cfg.geracao_backend == "hf" or cfg.fase == "geracao"
        self.model, self.hf, self.validacao = None, None, None
        t0 = time.time()
        if carregar_tl:
            from transformer_lens import HookedTransformer
            dtype = getattr(torch, cfg.dtype)
            kw = dict(device=self.device, dtype=dtype)
            if cfg.processar_pesos:
                kw.update(fold_ln=True, center_writing_weights=True, center_unembed=True, fold_value_biases=True)
            else:
                kw.update(fold_ln=False, center_writing_weights=False, center_unembed=False, fold_value_biases=False)
            registrar_modelo_tl(cfg.modelo, log)
            # O TransformerLens limita n_ctx de alguns modelos (p.ex. Llama-3.x) e usa n_ctx enorme em outros
            # (Qwen: 32768, que gasta GBs em máscaras). Fixamos o n_ctx antes de o modelo ser construído.
            import transformer_lens.loading_from_pretrained as lfp
            original = lfp.get_pretrained_model_config

            def com_n_ctx(*a, **k):
                c = original(*a, **k)
                if cfg.n_ctx and c.n_ctx != cfg.n_ctx:
                    c.n_ctx = cfg.n_ctx
                return c

            lfp.get_pretrained_model_config = com_n_ctx
            try:
                self.model = HookedTransformer.from_pretrained(cfg.modelo, **kw)
            finally:
                lfp.get_pretrained_model_config = original
            self.model.eval()
            self.model.requires_grad_(False)
            self.tok = self.model.tokenizer
            self.L, self.H, self.n_ctx = self.model.cfg.n_layers, self.model.cfg.n_heads, self.model.cfg.n_ctx
            log.info(f"TransformerLens: {cfg.modelo} em {time.time()-t0:.0f}s | camadas={self.L} cabeças={self.H} "
                     f"d_model={self.model.cfg.d_model} n_ctx={self.n_ctx} dtype={cfg.dtype} device={self.device}")
        if carregar_hf:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            extra = {"attn_implementation": "eager"} if "gemma-2" in cfg.modelo.lower() else {}
            self.hf = AutoModelForCausalLM.from_pretrained(cfg.modelo, torch_dtype=torch.bfloat16,
                                                           device_map=self.device, **extra)
            self.hf.eval()
            self.hf.requires_grad_(False)
            if self.model is None:
                self.tok = AutoTokenizer.from_pretrained(cfg.modelo)
                c = self.hf.config
                self.L, self.H = c.num_hidden_layers, c.num_attention_heads
                self.n_ctx = cfg.n_ctx
            log.info(f"Hugging Face (bf16) carregado | {time.time()-t0:.0f}s desde o início"
                     + (" | só HF (fase de geração)" if self.model is None else " | usado para gerar"))
        if self.n_ctx < 3000:
            log.warning(f"n_ctx={self.n_ctx} pode ser curto para OAB com histórico (etapa 3 com contexto).")
        self.eos_ids = self._eos_ids()
        self.letra_ids: dict[str, int] = {}
        self.variante_letra = None
        self.calibracao = None
        self.start_at_layer_ok = self._testar_start_at_layer() if self.model is not None else False
        if self.model is not None and self.hf is not None and cfg.validar_tl:
            self.validacao = self.validar_contra_hf()

    def validar_contra_hf(self, prompts: list[str] | None = None) -> dict:
        """Compara os logits da última posição no TransformerLens e no HF (bf16) para alguns prompts."""
        prompts = prompts or [
            "Qual é a capital do Brasil? Responda com uma palavra.",
            "Explique em uma frase o que é habeas corpus.",
            "A seguir há uma questão de múltipla escolha. Qual alternativa está correta: (A) 2+2=5 (B) 2+2=4? "
            "Responda apenas com a letra."]
        res = []
        for pr in prompts:
            ids, _ = self.tokenizar(self.render([{"role": "user", "content": pr}]))
            a = self.logits_ultimo(ids)[0].float()
            b = self.hf(input_ids=self._t(ids)).logits[0, -1].float()
            n = min(a.shape[-1], b.shape[-1])
            a, b = a[:n] - a[:n].mean(), b[:n] - b[:n].mean()
            top = torch.topk(b, 20).indices
            res.append({"corr": float(torch.corrcoef(torch.stack([a, b]))[0, 1]),
                        "top1_igual": int(a.argmax()) == int(b.argmax()),
                        "max_dif_top20": float((a[top] - b[top]).abs().max())})
        out = {"corr_min": min(r["corr"] for r in res), "top1_concordancia": sum(r["top1_igual"] for r in res) / len(res),
               "max_dif_top20": max(r["max_dif_top20"] for r in res)}
        ok = out["corr_min"] > 0.99 and out["top1_concordancia"] >= 2 / 3
        (self.log.info if ok else self.log.warning)(
            f"Validação TL × HF: corr mínima {out['corr_min']:.4f} | top-1 igual em {100*out['top1_concordancia']:.0f}% "
            f"| maior diferença no top-20 {out['max_dif_top20']:.2f} logits" + ("" if ok else " — VERIFICAR antes de rodar"))
        out["ok"] = ok
        return out

    # ---------------- utilidades de tokenização ----------------
    def _eos_ids(self) -> list[int]:
        ids = set()
        if self.tok.eos_token_id is not None:
            ids.add(int(self.tok.eos_token_id))
        for t in ["<|eot_id|>", "<|end_of_text|>", "<end_of_turn>", "<|im_end|>", "<|endoftext|>", "<eos>", "</s>"]:
            i = self.tok.convert_tokens_to_ids(t)
            if isinstance(i, int) and i >= 0 and i != self.tok.unk_token_id:
                ids.add(i)
        return sorted(ids)

    def render(self, msgs: list[dict], prefill: str = "", gerar=True) -> str:
        return self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=gerar) + prefill

    def tokenizar(self, texto: str) -> tuple[list, list]:
        enc = self.tok(texto, add_special_tokens=False, return_offsets_mapping=True)
        return list(enc["input_ids"]), [tuple(o) for o in enc["offset_mapping"]]

    def preparar(self, msgs: list[dict], questao_txt: str | None, prefill: str = "") -> PromptTok:
        texto = self.render(msgs, prefill)
        ids, offs = self.tokenizar(texto)
        if len(ids) >= self.n_ctx:
            raise RuntimeError(f"prompt com {len(ids)} tokens excede n_ctx={self.n_ctx}")
        Q, qc = [], (-1, -1)
        if questao_txt:
            ini = texto.rfind(questao_txt)
            if ini >= 0:
                qc = (ini, ini + len(questao_txt))
                Q = [i for i, (a, b) in enumerate(offs) if b > a and a < qc[1] and b > qc[0]]
        return PromptTok(texto, ids, offs, qc, Q)

    def _t(self, ids) -> torch.Tensor:
        return torch.tensor([ids], dtype=torch.long, device=self.device)

    # ---------------- letras ----------------
    def lote_para(self, P: int, maximo: int) -> int:
        """Maior lote cujo tensor de escores de atenção [lote, H, P, P] em fp32 cabe no orçamento."""
        por_amostra = self.H * P * P * 4
        return max(1, min(int(maximo), int(self.cfg.orcamento_atencao_gb * 1e9 // por_amostra)))

    def calibrar_letras(self, textos_prompt: list[str]):
        """Escolhe a variante do token da letra (com ou sem espaço) com mais massa após o prefill."""
        variantes = {}
        for nome, fmt in [("sem_espaco", "{}"), ("com_espaco", " {}")]:
            ids, unico = {}, True
            for l in LETRAS:
                enc = self.tok.encode(fmt.format(l), add_special_tokens=False)
                unico &= len(enc) == 1
                ids[l] = enc[0] if len(enc) == 1 else enc[-1]
            variantes[nome] = (ids, unico)
        massas = {n: [] for n in variantes}
        for tx in textos_prompt:
            ids, _ = self.tokenizar(tx)
            p = self.logits_ultimo(ids)[0].float().softmax(-1)
            for n, (lid, _) in variantes.items():
                massas[n].append(float(sum(p[lid[l]] for l in "ABCD")))
        medias = {n: float(np.mean(v)) for n, v in massas.items()}
        self.variante_letra = max(medias, key=medias.get)
        self.letra_ids, unico = variantes[self.variante_letra]
        self.calibracao = {"variante": self.variante_letra, "massa_media": medias, "token_unico": unico,
                           "ids": self.letra_ids,
                           "tokens": {l: self.tok.convert_ids_to_tokens(i) for l, i in self.letra_ids.items()}}
        self.log.info(f"Calibração das letras: {self.calibracao}")
        if not unico:
            self.log.warning("Alguma letra não é token único nesta variante; usando o primeiro/último token.")
        if medias[self.variante_letra] < 0.5:
            self.log.warning(f"Massa nas letras após o prefill é baixa ({medias[self.variante_letra]:.2f}). "
                             "Revise prefill_resposta antes da rodada completa.")
        return self.calibracao

    def logits_letras(self, vec: torch.Tensor, letras: str) -> dict:
        return {l: float(vec[self.letra_ids[l]]) for l in letras}

    def ld(self, vec: torch.Tensor, alvo: str, orig: str) -> float:
        return float(vec[self.letra_ids[alvo]] - vec[self.letra_ids[orig]])

    # ---------------- forward ----------------
    def _cabeca(self, resid_final: torch.Tensor) -> torch.Tensor:
        """ln_final + unembed só na última posição (economiza a matriz de logits inteira)."""
        lg = self.model.unembed(self.model.ln_final(resid_final[:, -1:, :]))[:, 0, :]
        cap = getattr(self.model.cfg, "output_logits_soft_cap", -1.0) or -1.0
        if cap > 0:
            lg = cap * torch.tanh(lg / cap)
        return lg

    def logits_ultimo(self, ids, fwd_hooks=(), lote: int = 1) -> torch.Tensor:
        toks = self._t(ids)
        if lote > 1:
            toks = toks.expand(lote, -1)
        if self.model is None:                       # fase de geração: só HF, sem hooks
            if fwd_hooks:
                raise RuntimeError("hooks exigem o TransformerLens (fase != 'geracao')")
            return self.hf(input_ids=toks).logits[:, -1, :].float()
        with self.model.hooks(fwd_hooks=list(fwd_hooks)):
            resid = self.model(toks, stop_at_layer=self.L)
            return self._cabeca(resid).float()

    def a_partir_da_camada(self, resid: torch.Tensor, camada: int, ids=None) -> torch.Tensor:
        """Roda do bloco `camada` até o fim, a partir de resid_pre[camada]. resid: [b, pos, d]."""
        if self.start_at_layer_ok:
            out = self.model(resid, start_at_layer=camada, stop_at_layer=self.L)
            return self._cabeca(out).float()
        nome = f"blocks.{camada}.hook_resid_pre"

        def substitui(act, hook):
            return resid.to(act.dtype)

        toks = self._t(ids).expand(resid.shape[0], -1)
        with self.model.hooks(fwd_hooks=[(nome, substitui)]):
            out = self.model(toks, stop_at_layer=self.L)
        return self._cabeca(out).float()

    def _testar_start_at_layer(self) -> bool:
        try:
            ids, _ = self.tokenizar("Teste rápido de consistência do TransformerLens.")
            c = max(1, self.L // 2)
            cap = {}
            nome = f"blocks.{c}.hook_resid_pre"

            def guarda(act, hook):
                cap["r"] = act.detach().clone()

            ref = self.logits_ultimo(ids, fwd_hooks=[(nome, guarda)])
            out = self.model(cap["r"], start_at_layer=c, stop_at_layer=self.L)
            dif = float((self._cabeca(out).float() - ref).abs().max())
            ok = dif < (1e-2 if self.cfg.dtype == "float32" else 0.5)
            self.log.info(f"start_at_layer: diferença máxima de logits {dif:.2e} -> {'usando' if ok else 'NÃO usando'}")
            return ok
        except Exception as e:
            self.log.info(f"start_at_layer indisponível ({type(e).__name__}: {e}); usando hook de substituição")
            return False

    def gradiente_resid(self, ids, a: int, b: int) -> tuple[float, torch.Tensor, torch.Tensor]:
        """LD = ℓ_a − ℓ_b na última posição; devolve (LD, resid_pre [L, T, d], dLD/dresid_pre [L, T, d]).
        Usa torch.utils.checkpoint por bloco, para o backward caber na memória mesmo em fp32."""
        import torch.utils.checkpoint as ckpt
        m = self.model
        with torch.no_grad():
            try:
                r0 = m(self._t(ids), stop_at_layer=0)                # resid_pre da camada 0
            except Exception:
                r0 = self.capturar(ids, ["blocks.0.hook_resid_pre"])[1]["blocks.0.hook_resid_pre"][None]
        with torch.enable_grad():
            x = r0.detach().requires_grad_(True)
            entradas = [x]
            for l in range(self.L):
                x = ckpt.checkpoint(m.blocks[l], x, use_reentrant=False)
                if l + 1 < self.L:
                    x.retain_grad()
                    entradas.append(x)
            lg = self._cabeca(x).float()
            ld = lg[0, a] - lg[0, b]
            ld.backward()
        resid = torch.stack([e[0].detach() for e in entradas])
        grads = torch.stack([e.grad[0].detach() for e in entradas])
        val = float(ld.detach())
        del entradas, x, lg, ld
        return val, resid, grads

    def capturar(self, ids, nomes: list[str]) -> tuple[torch.Tensor, dict]:
        cache = {}

        def h(act, hook):
            cache[hook.name] = act[0].detach().clone()

        lg = self.logits_ultimo(ids, fwd_hooks=[(n, h) for n in nomes])
        return lg[0], cache

    def resid_todas(self, ids) -> tuple[torch.Tensor, torch.Tensor]:
        nomes = [f"blocks.{l}.hook_resid_pre" for l in range(self.L)]
        lg, cache = self.capturar(ids, nomes)
        return lg, torch.stack([cache[n] for n in nomes])  # [L, pos, d]

    def logits_completos(self, ids) -> torch.Tensor:
        return self.model(self._t(ids), return_type="logits")[0].float()

    def nll_media(self, ids, ini, fim, logits=None) -> float:
        if logits is None:
            logits = self.logits_completos(ids)
        ini = max(ini, 1)
        lp = logits[ini - 1: fim - 1].log_softmax(-1)
        alvo = torch.tensor(ids[ini:fim], device=lp.device)
        return float(-lp.gather(1, alvo[:, None]).mean())

    # ---------------- geração ----------------
    def gerar(self, msgs, max_novos: int, T: float, semente: int, prefill: str = "") -> dict:
        texto = self.render(msgs, prefill)
        ids, _ = self.tokenizar(texto)
        lim = self.n_ctx - len(ids) - 1 if self.hf is None else max_novos
        max_novos = int(min(max_novos, lim))
        if max_novos <= 0:
            raise RuntimeError(f"sem espaço de contexto para gerar ({len(ids)} tokens de prompt)")
        torch.manual_seed(semente)
        t0 = time.time()
        if self.hf is not None:
            inp = self._t(ids)
            kw = dict(max_new_tokens=max_novos, eos_token_id=self.eos_ids, pad_token_id=self.eos_ids[0],
                      attention_mask=torch.ones_like(inp))
            if T > 0:   # amostragem só por temperatura, como no TransformerLens (sem o top_p/top_k do modelo)
                kw.update(do_sample=True, temperature=float(T), top_p=1.0, top_k=0)
            else:
                kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
            out = self.hf.generate(inp, **kw)
            novos = out[0, len(ids):].tolist()
        else:
            out = self.model.generate(self._t(ids), max_new_tokens=max_novos, do_sample=T > 0,
                                      temperature=T if T > 0 else 1.0, stop_at_eos=True,
                                      eos_token_id=self.eos_ids, use_past_kv_cache=True, verbose=False,
                                      return_type="tokens")
            novos = out[0, len(ids):].tolist()
        terminou = any(t in self.eos_ids for t in novos)
        if terminou:
            novos = novos[: next(i for i, t in enumerate(novos) if t in self.eos_ids)]
        return {"texto": self.tok.decode(novos, skip_special_tokens=True), "n_tokens": len(novos),
                "truncada": not terminou, "segundos": time.time() - t0, "n_tokens_prompt": len(ids)}

    def responder_letra(self, pt: PromptTok, letras: str, T: float, semente: int) -> dict:
        """Resposta sem CoT: letra na posição após o prefill. T=0: argmax entre as letras."""
        vec = self.logits_ultimo(pt.ids)[0]
        p = vec.softmax(-1)
        lg = self.logits_letras(vec, letras)
        top1 = int(vec.argmax())
        if T > 0:
            g = torch.Generator().manual_seed(semente)
            pl = torch.tensor([lg[l] for l in letras]).div(T).softmax(-1)
            letra = letras[int(torch.multinomial(pl, 1, generator=g))]
        else:
            letra = max(lg, key=lg.get)
        return {"letra": letra, "logits_letras": lg, "vec": vec,
                "massa_letras": float(sum(p[self.letra_ids[l]] for l in letras)),
                "top1_token": self.tok.convert_ids_to_tokens(top1),
                "top1_eh_letra": top1 in [self.letra_ids[l] for l in letras]}


# =============================================================================
# ETAPA 4A — PATCHING x_CE -> x
# =============================================================================


def patching_edicao(ex: Executor, cfg: Config, px: PromptTok, pce: PromptTok, letras_x: tuple, letras_ce: tuple,
                    ld_x: float, ld_ce: float, chave: str) -> tuple[dict, list, dict]:
    """Retorna (métricas, blocos, arrays para o npz)."""
    L = ex.L
    a_x, b_x = (ex.letra_ids[letras_x[0]], ex.letra_ids[letras_x[1]])
    a_c, b_c = (ex.letra_ids[letras_ce[0]], ex.letra_ids[letras_ce[1]])
    den = ld_ce - ld_x
    al = alinhar_tokens(px.ids, pce.ids, cfg.juntar_blocos_gap)
    transplante = al["transplante"]
    t_resp, s_resp = len(px.ids) - 1, len(pce.ids) - 1
    transplante[t_resp] = s_resp
    E = sorted({b["t"] for b in al["blocos"]})
    Q = px.Q
    Tx = len(px.ids)
    met = {"n_blocos": len(al["blocos"]), "n_E_total": len(E)}

    # ---- caches -----------------------------------------------------------
    t0 = time.time()
    _, cache_ce = ex.resid_todas(pce.ids)                          # [L, Tce, d]
    # ---- atribuição (gradiente de LD em x em todas as camadas/posições) ----
    # Gradiente com checkpoint por bloco: no backward cada bloco é recomputado, e só um bloco por vez
    # guarda tensores. Sem isso, o autograd guardava cópias rearranjadas dos pesos de todas as camadas
    # (~+30 GB em fp32), o que estourava a A100 no Llama.
    ld_val, resid_x, G = ex.gradiente_resid(px.ids, a_x, b_x)          # [L, Tx, d] cada
    G = G.to(resid_x.dtype)
    if abs(ld_val - ld_x) > (1e-2 if cfg.dtype == "float32" else 0.25):
        ex.log.warning(f"{chave}: LD pelo caminho do gradiente ({ld_val:.4f}) difere do da etapa 1 ({ld_x:.4f})")
    met["ld_x_recalc"] = ld_val
    T_idx = torch.tensor(sorted(transplante), device=resid_x.device)
    S_idx = torch.tensor([transplante[int(t)] for t in T_idx.tolist()], device=resid_x.device)
    r_attr = torch.full((L, Tx), float("nan"))
    dif = cache_ce[:, S_idx, :] - resid_x[:, T_idx, :]
    r_attr[:, T_idx.cpu()] = ((dif * G[:, T_idx, :]).sum(-1).float() / den).cpu()
    del G, dif
    met["seg_atribuicao"] = time.time() - t0

    # ---- seleção de pontos para patching exato ------------------------------
    alvo_pos = [t for t in E if t in transplante] + [t_resp]
    pontos = {(l, t) for l in range(L) for t in alvo_pos}
    candidatos = [(l, t) for l in range(L) for t in Q if t in transplante and (l, t) not in pontos]
    if cfg.modo_patching == "exato":
        pontos |= set(candidatos)
    elif candidatos:
        sc = np.array([abs(float(r_attr[l, t])) for l, t in candidatos])
        k = min(cfg.max_pontos_top, max(1, int(round(cfg.fracao_top_attr * len(candidatos)))))
        for i in np.argsort(-sc)[:k]:
            pontos.add(candidatos[i])
    met["n_pontos_exatos"] = len(pontos)

    # ---- patching exato (denoising: limpo -> corrompido) ---------------------
    t0 = time.time()
    r_ex = torch.full((L, Tx), float("nan"))
    por_camada: dict[int, list] = {}
    for l, t in sorted(pontos):
        por_camada.setdefault(l, []).append(t)
    for l, ts in por_camada.items():
        passo = ex.lote_para(Tx, cfg.lote_patching)
        for i in range(0, len(ts), passo):
            lote = ts[i: i + passo]
            base = resid_x[l].unsqueeze(0).repeat(len(lote), 1, 1)
            rows = torch.arange(len(lote), device=base.device)
            tt = torch.tensor(lote, device=base.device)
            ss = torch.tensor([transplante[t] for t in lote], device=base.device)
            base[rows, tt] = cache_ce[l, ss].to(base.dtype)
            lg = ex.a_partir_da_camada(base, l, px.ids)
            r_ex[l, lote] = ((lg[:, a_x] - lg[:, b_x] - ld_x) / den).cpu()
    met["seg_patching_exato"] = time.time() - t0
    r_comb = torch.where(torch.isnan(r_ex), r_attr, r_ex)

    # ---- noising (direção inversa: corrompido -> limpo, só E e resposta) ----
    r_noise = torch.full((L, Tx), float("nan"))
    if cfg.rodar_noising:
        t0 = time.time()
        for l in range(L):
            passo = ex.lote_para(len(pce.ids), cfg.lote_patching)
            for i in range(0, len(alvo_pos), passo):
                lote = alvo_pos[i: i + passo]
                base = cache_ce[l].unsqueeze(0).repeat(len(lote), 1, 1)
                rows = torch.arange(len(lote), device=base.device)
                ss = torch.tensor([transplante[t] for t in lote], device=base.device)
                tt = torch.tensor(lote, device=base.device)
                base[rows, ss] = resid_x[l, tt].to(base.dtype)
                lg = ex.a_partir_da_camada(base, l, pce.ids)
                r_noise[l, lote] = ((ld_ce - (lg[:, a_c] - lg[:, b_c])) / den).cpu()
        met["seg_noising"] = time.time() - t0

    rc = r_comb.numpy()
    cea = calcular_cea(rc, Q, E, cfg.n_permutacoes, semente_de(cfg.semente, chave, "cea"), "max")
    cea_soma = calcular_cea(rc, Q, E, 0, 0, "soma")
    e_por_E = cea.pop("e_por_posicao_E", {})
    cea_soma.pop("e_por_posicao_E", None)
    met.update(cea)
    met["cea_soma"] = cea_soma["cea"]
    met["cea_estrela_soma"] = cea_soma["cea_estrela"]
    met.update(perfil_camadas(rc[:, t_resp]))
    # concordância entre direções no subespaço exato (E × camadas)
    if E:
        a = np.nan_to_num(rc[:, E].ravel())
        b = np.nan_to_num(r_noise.numpy()[:, E].ravel())
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        met["cos_denoise_noise_E"] = float(a @ b / (na * nb)) if na > 0 and nb > 0 else float("nan")
        met["noising_max_E"] = float(np.nanmax(r_noise.numpy()[:, E])) if np.isfinite(r_noise.numpy()[:, E]).any() else float("nan")
    # correlação atribuição × exato (diagnóstico da aproximação linear)
    msk = ~torch.isnan(r_ex)
    if int(msk.sum()) > 3:
        met["corr_attr_exato"] = float(np.corrcoef(r_attr[msk].numpy(), r_ex[msk].numpy())[0, 1])

    blocos = []
    for i, b in enumerate(al["blocos"]):
        blocos.append({**b, "bloco": i,
                       "texto_x": ex.tok.decode(px.ids[b["x_ini"]: b["x_fim"]]),
                       "texto_ce": ex.tok.decode(pce.ids[b["ce_ini"]: b["ce_fim"]]),
                       "em_Q": b["t"] in set(Q), "e_t": e_por_E.get(b["t"], float("nan"))})
    arrays = {"ids_x": np.array(px.ids), "ids_ce": np.array(pce.ids), "Q": np.array(Q), "E": np.array(E),
              "transplante_t": T_idx.cpu().numpy(), "transplante_s": S_idx.cpu().numpy(),
              "r_attr": r_attr.numpy(), "r_exato": r_ex.numpy(), "r_comb": rc, "r_noising": r_noise.numpy(),
              "t_resp": np.array(t_resp),
              "tokens_x": np.array([ex.tok.decode([i]) for i in px.ids], dtype="U"),
              "tokens_ce": np.array([ex.tok.decode([i]) for i in pce.ids], dtype="U")}
    del cache_ce, resid_x
    return met, blocos, arrays


def ert_blocos(ex: Executor, cfg: Config, px: PromptTok, pce: PromptTok, letras_ce: tuple, ld_x: float, ld_ce: float,
               blocos: list) -> list:
    """ERT_b: reverte cada bloco editado (em tokens) ao texto original e mede a perda de LD; fluência relativa."""
    if not blocos:
        return blocos
    den = ld_ce - ld_x
    qa, qb = (min(pce.Q), max(pce.Q) + 1) if pce.Q else (1, len(pce.ids) - 1)
    lg_ce = ex.logits_completos(pce.ids)
    nll_ce = ex.nll_media(pce.ids, qa, qb, lg_ce)
    del lg_ce
    for b in blocos[: cfg.max_blocos_ert]:
        rev = pce.ids[: b["ce_ini"]] + px.ids[b["x_ini"]: b["x_fim"]] + pce.ids[b["ce_fim"]:]
        lg = ex.logits_completos(rev)
        ld_rev = ex.ld(lg[-1], *letras_ce)
        delta_len = (b["x_fim"] - b["x_ini"]) - (b["ce_fim"] - b["ce_ini"])
        nll_rev = ex.nll_media(rev, qa, qb + delta_len, lg)
        b.update(ld_rev=ld_rev, ert=(ld_ce - ld_rev) / den, nll_rev=nll_rev, nll_ce=nll_ce,
                 ppl_rel=float(math.exp(nll_rev - nll_ce)))
        del lg
    return blocos


# =============================================================================
# ETAPA 4B — MEDIAÇÃO PELO CONTEXTO
# =============================================================================


def mediacao_contexto(ex: Executor, cfg: Config, pctx: PromptTok, hist_render: str, pedido_sce: str,
                      alvo_spec: str, letra_alvo_pedido: str, letras_ce: tuple, ld_sem: float, ld_ctx: float,
                      ids_sem: list, msgs_outro: list | None, questao_ce_txt: str, letra_alvo_outro: str | None) -> tuple[dict, list, dict]:
    L, H = ex.L, ex.H
    a, b = ex.letra_ids[letras_ce[0]], ex.letra_ids[letras_ce[1]]
    P = len(pctx.ids)
    offs = pctx.offsets
    met = {"cmr_incluido": bool(ld_ctx - ld_sem >= cfg.tau), "ganho_ld_ctx": ld_ctx - ld_sem}

    # posições: histórico (chaves bloqueadas) e nova pergunta + resposta (consultas)
    if pctx.texto.startswith(hist_render):
        fim_hist = len(hist_render)
    else:
        fim_hist = pctx.texto.rfind("A seguir há uma questão")
        ex.log.warning("render(H) não é prefixo do prompt com contexto; usando início da última instrução")
    q_ini = next((i for i, (s, e) in enumerate(offs) if s >= fim_hist and e > s), None)
    if q_ini is None:
        raise RuntimeError("não foi possível localizar o fim do histórico no prompt com contexto")
    chaves_hist = [i for i in range(1, q_ini)]                          # preserva o primeiro token (sumidouro)
    ini_pedido = pctx.texto.rfind(pedido_sce, 0, fim_hist)
    ini_alvo = pctx.texto.find(alvo_spec, ini_pedido) if ini_pedido >= 0 else -1
    if ini_alvo >= 0:
        sp = (ini_alvo, ini_alvo + len(alvo_spec))
        chaves_alvo = [i for i, (s, e) in enumerate(offs) if e > s and s < sp[1] and e > sp[0]]
        c_letra = ini_alvo + 1  # "(" + letra
        pos_letra = [i for i, (s, e) in enumerate(offs) if s <= c_letra < e]
    else:
        chaves_alvo, pos_letra = [], []
        ex.log.warning("menção ao alvo não localizada no pedido de revisão; CMR-alvo indisponível")
    # declarações do alvo ('Resposta: C') na resposta do próprio modelo à etapa 2 (depois do pedido)
    ini_turno_sce = ini_pedido + len(pedido_sce) if ini_pedido >= 0 else fim_hist
    trecho = pctx.texto[ini_turno_sce:fim_hist]
    spans = [(ini_turno_sce + s, ini_turno_sce + e)
             for l_ in {letra_alvo_pedido, letras_ce[0]} for s, e in spans_declaracao(trecho, l_)]
    chaves_decl = sorted({i for i, (s, e) in enumerate(offs) if e > s and any(s < b2 and e > a2 for a2, b2 in spans)})
    met.update(n_tokens_ctx=P, n_chaves_hist=len(chaves_hist), n_chaves_alvo=len(chaves_alvo),
               n_chaves_decl=len(chaves_decl), n_declaracoes=len(spans))

    # ---- knockouts --------------------------------------------------------
    # "direto": só as posições da nova pergunta e da resposta deixam de ver as chaves bloqueadas.
    # "prop":   NENHUMA posição posterior vê as chaves bloqueadas, inclusive as do próprio histórico. Isso impede
    #           que a informação do trecho chegue à resposta indiretamente, por tokens do histórico que já a leram
    #           (p.ex. a SCE gerada, que foi escrita depois do pedido e carrega o alvo no fluxo residual).
    ini_turno_pedido = ini_pedido if ini_pedido >= 0 else fim_hist

    def toks_em(a_, b_):
        return [i for i, (s, e) in enumerate(offs) if i > 0 and e > s and s < b_ and e > a_]

    alvo_decl = sorted(set(chaves_alvo) | set(chaves_decl))
    variantes = [  # nome, modo, chaves, vazio_significa
        ("total", "direto", chaves_hist, "nan"),
        ("alvo", "direto", chaves_alvo, "nan"),
        ("decl", "direto", chaves_decl, "zero"),
        ("alvo_decl", "direto", alvo_decl, "nan"),
        ("alvo_prop", "prop", chaves_alvo, "nan"),
        ("decl_prop", "prop", chaves_decl, "zero"),
        ("alvo_decl_prop", "prop", alvo_decl, "nan"),
        ("seg_q1", "prop", toks_em(0, ini_turno_pedido), "nan"),          # questão original + 'Resposta: ŷ'
        ("seg_pedido", "prop", toks_em(ini_turno_pedido, ini_turno_sce), "nan"),
        ("seg_sce", "prop", toks_em(ini_turno_sce, fim_hist), "nan"),      # a resposta do modelo à etapa 2
    ]
    met.update({f"n_chaves_{n}": len(k) for n, _, k, _ in variantes})
    ar = torch.arange(P, device=ex.device)
    den = ld_ctx - ld_sem
    calc = [v for v in variantes if v[2]]
    passo = ex.lote_para(P, 4)
    met["lote_knockout"] = passo
    for i0 in range(0, len(calc), passo):
        lote_v = calc[i0: i0 + passo]
        M = torch.zeros(len(lote_v), 1, P, P, dtype=torch.bool, device=ex.device)
        for j, (_, modo, K, _) in enumerate(lote_v):
            Kt = torch.tensor(K, device=ex.device)
            if modo == "direto":
                M[j, 0, q_ini:][:, K] = True
            else:
                M[j, 0][:, Kt] = ar[:, None] > Kt[None, :]

        def ko(scores, hook, M=M):
            return scores.masked_fill(M, float("-inf"))

        lg = ex.logits_ultimo(pctx.ids, fwd_hooks=[(f"blocks.{l}.attn.hook_attn_scores", ko) for l in range(L)],
                              lote=len(lote_v))
        for j, (nome, _, _, _) in enumerate(lote_v):
            ld_ko = float(lg[j, a] - lg[j, b])
            met[f"ld_ctx_ko_{nome}"] = ld_ko
            met[f"cmr_{nome}"] = (ld_ctx - ld_ko) / den if den != 0 else float("nan")
        del M, lg
    for nome, _, K, vazio in variantes:
        if not K:
            met[f"ld_ctx_ko_{nome}"] = ld_ctx if vazio == "zero" else float("nan")
            met[f"cmr_{nome}"] = 0.0 if vazio == "zero" else float("nan")

    # ---- DLA por cabeça + atenção ao token da letra-alvo no histórico -------
    zs, pats, esc = {}, {}, {}

    def gz(z, hook):
        zs[hook.layer()] = z[0, -1].detach().clone()           # [H, d_head]

    def gp(p, hook):
        pats[hook.layer()] = p[0, :, -1, :].detach().clone()   # [H, P]

    def gs(s, hook):
        esc["s"] = s[0, -1].detach().clone()

    hooks = [(f"blocks.{l}.attn.hook_z", gz) for l in range(L)] + \
            [(f"blocks.{l}.attn.hook_pattern", gp) for l in range(L)] + [("ln_final.hook_scale", gs)]
    lg = ex.logits_ultimo(pctx.ids, fwd_hooks=hooks)
    met["ld_ctx_recalc"] = float(lg[0, a] - lg[0, b])
    W_U = ex.model.W_U
    direcao = (W_U[:, a] - W_U[:, b]).float()
    escala = esc["s"].float().reshape(-1)[0] if esc else torch.tensor(1.0)
    dla = torch.zeros(L, H)
    att = torch.zeros(L, H)
    for l in range(L):
        o = torch.einsum("hd,hdm->hm", zs[l].float(), ex.model.blocks[l].attn.W_O.float())
        dla[l] = (o @ direcao / escala).cpu()
        if pos_letra:
            att[l] = pats[l][:, pos_letra].sum(-1).float().cpu()
    del pats

    # z sem contexto (para ablação) na última posição
    zs_ctx = zs
    zs = {}
    ex.logits_ultimo(ids_sem, fwd_hooks=[(f"blocks.{l}.attn.hook_z", gz) for l in range(L)])
    z_sem = zs
    zs = zs_ctx

    ordem = torch.argsort(dla.flatten(), descending=True).tolist()
    top = [(i // H, i % H) for i in ordem[: cfg.top_k_cabecas]]
    candidatas = [(l, h) for (l, h) in top if float(att[l, h]) >= cfg.limiar_atencao_alvo]
    conjuntos = [[c] for c in candidatas] + ([candidatas] if len(candidatas) > 1 else [])
    ablacao = {}
    if conjuntos:
        passo = ex.lote_para(P, cfg.lote_patching)
        for i in range(0, len(conjuntos), passo):
            lote = conjuntos[i: i + passo]
            por_camada: dict[int, list] = {}
            for row, conj in enumerate(lote):
                for (l, h) in conj:
                    por_camada.setdefault(l, []).append((row, h))

            def faz_hook(l):
                def ab(z, hook):
                    for row, h in por_camada[l]:
                        z[row, -1, h] = z_sem[l][h].to(z.dtype)
                    return z
                return ab

            lg = ex.logits_ultimo(pctx.ids, fwd_hooks=[(f"blocks.{l}.attn.hook_z", faz_hook(l)) for l in por_camada],
                                  lote=len(lote))
            for row, conj in enumerate(lote):
                ablacao[tuple(conj)] = float(lg[row, a] - lg[row, b])
    met["n_cabecas_candidatas"] = len(candidatas)
    if len(candidatas) > 1:
        met["ld_ctx_ablacao_conjunta"] = ablacao[tuple(candidatas)]
        met["queda_ablacao_conjunta"] = ld_ctx - ablacao[tuple(candidatas)]
    elif len(candidatas) == 1:
        met["ld_ctx_ablacao_conjunta"] = ablacao[(candidatas[0],)]
        met["queda_ablacao_conjunta"] = ld_ctx - ablacao[(candidatas[0],)]
    cabecas = []
    for rank, (l, h) in enumerate(top):
        cab = {"camada": l, "cabeca": h, "rank_dla": rank, "dla": float(dla[l, h]), "att_alvo": float(att[l, h]),
               "candidata": (l, h) in candidatas}
        if (l, h) in candidatas:
            cab["ld_ablado"] = ablacao[((l, h),)]
            cab["queda_ablacao"] = ld_ctx - cab["ld_ablado"]
        cabecas.append(cab)

    # ---- controle: histórico de OUTRA questão --------------------------------
    if msgs_outro is not None:
        try:
            po = ex.preparar(msgs_outro, questao_ce_txt, ex.cfg.prefill_resposta)
            lo = ex.logits_ultimo(po.ids)[0]
            met["ld_ctx_outro_hist"] = ex.ld(lo, *letras_ce)
            met["outro_hist_alvo_igual"] = letra_alvo_outro == letras_ce[0]
            # fração do ganho de contexto reproduzida por um histórico de revisão de OUTRA questão
            met["frac_ganho_outro_hist"] = ((met["ld_ctx_outro_hist"] - ld_sem) / (ld_ctx - ld_sem)
                                            if ld_ctx != ld_sem else float("nan"))
        except Exception as e:
            met["ld_ctx_outro_hist_erro"] = str(e)[:200]
    arrays = {"dla": dla.numpy(), "att_alvo": att.numpy(), "chaves_alvo": np.array(chaves_alvo), "chaves_decl": np.array(chaves_decl),
              "pos_letra_alvo": np.array(pos_letra), "q_ini_ctx": np.array(q_ini)}
    return met, cabecas, arrays


# =============================================================================
# PIPELINE POR ITEM
# =============================================================================


class Experimento:
    def __init__(self, cfg: Config, executor: Executor | None = None):
        self.cfg = cfg
        self.run_dir, self.log = preparar_saida(cfg)
        self.ex = executor or Executor(cfg, self.log)
        self.ex.log = self.log
        self.ex.cfg = cfg
        self.lingua = DetectorLingua(cfg.detector_lingua, self.log)
        self.p_itens = os.path.join(self.run_dir, "itens.jsonl")
        self.p_erros = os.path.join(self.run_dir, "erros.jsonl")
        self.hist_anterior: dict[str, tuple] = {}
        # retomada (p.ex. fase 2 dos modelos grandes): reaproveita a calibração das letras e o prefill do run
        p_amb = os.path.join(self.run_dir, "ambiente.json")
        if os.path.exists(p_amb) and not self.ex.letra_ids:
            try:
                calr = json.load(open(p_amb, encoding="utf-8")).get("calibracao_letras") or {}
                if calr.get("ids"):
                    self.ex.letra_ids = {k: int(v) for k, v in calr["ids"].items()}
                    self.ex.calibracao = calr
                    cfg.prefill_resposta = calr.get("prefill_escolhido") or cfg.prefill_resposta
                    self.log.info(f"Calibração do run reaproveitada: prefill {cfg.prefill_resposta!r}, letras {self.ex.letra_ids}")
            except Exception as e:
                self.log.warning(f"não foi possível ler a calibração anterior: {e}")
        try:  # cópia do código usado no run, para reprodutibilidade
            import shutil
            shutil.copy(os.path.abspath(__file__), os.path.join(self.run_dir, "codigo_sce_ptbr.py"))
        except Exception:
            pass
        self._salvar_ambiente()

    def _salvar_ambiente(self):
        import importlib.metadata as md
        versoes = {}
        for p in ["torch", "transformer_lens", "transformers", "numpy", "pandas", "rapidfuzz", "fasttext", "langdetect"]:
            try:
                versoes[p] = md.version(p)
            except Exception:
                versoes[p] = None
        amb = {"python": platform.python_version(), "versoes": versoes, "detector_lingua": self.lingua.nome,
               "gpu": torch.cuda.get_device_name(0) if torch is not None and torch.cuda.is_available() else None,
               "modelo": self.cfg.modelo, "tokenizador": getattr(self.ex.tok, "name_or_path", None),
               "n_camadas": self.ex.L, "n_cabecas": self.ex.H, "n_ctx": self.ex.n_ctx,
               "start_at_layer": self.ex.start_at_layer_ok, "eos_ids": self.ex.eos_ids,
               "fase": self.cfg.fase, "validacao_tl_hf": getattr(self.ex, "validacao", None),
               "calibracao_letras": getattr(self.ex, "calibracao", None),
               "inicio": datetime.datetime.now().isoformat()}
        json.dump(amb, open(os.path.join(self.run_dir, "ambiente.json"), "w", encoding="utf-8"),
                  ensure_ascii=False, indent=2, default=_json_default)

    # ---------------------------------------------------------------------
    def calibrar(self, questoes: list[Questao]) -> dict:
        """Escolhe o prefill (entre cfg.prefills_candidatos) e a variante do token da letra."""
        ex, cfg = self.ex, self.cfg
        cands = list(cfg.prefills_candidatos) or [cfg.prefill_resposta]
        msgs = [[{"role": "user", "content": prompt_etapa1(q.texto, q.dataset, "irrestrita")}] for q in questoes[:5]]
        res = {}
        for pf in cands:
            cal = ex.calibrar_letras([ex.render(m, pf) for m in msgs])
            res[pf] = cal["massa_media"][cal["variante"]]
        escolhido = next((pf for pf in cands if res[pf] >= cfg.limiar_massa_letras), max(res, key=res.get))
        cfg.prefill_resposta = escolhido
        cal = ex.calibrar_letras([ex.render(m, escolhido) for m in msgs])
        cal["prefill_escolhido"] = escolhido
        cal["massa_por_prefill"] = res
        ex.calibracao = cal
        self.log.info(f"Prefill escolhido: {escolhido!r} | massa nas letras por prefill: "
                      + ", ".join(f"{k!r}={v:.2f}" for k, v in res.items()))
        json.dump(cfg.to_dict(), open(os.path.join(self.run_dir, "config.json"), "w", encoding="utf-8"),
                  ensure_ascii=False, indent=2)
        self._salvar_ambiente()
        return cal

    def diagnostico(self, questoes: list[Questao]) -> dict:
        """Checagens do piloto antes de rodar: letras, prefill, recorte de Q, tamanho dos prompts."""
        ex, cfg = self.ex, self.cfg
        cal = self.calibrar(questoes)
        q = questoes[0]
        pt = ex.preparar([{"role": "user", "content": prompt_etapa1(q.texto, q.dataset, "irrestrita")}],
                         q.texto, cfg.prefill_resposta)
        vec = ex.logits_ultimo(pt.ids)[0]
        top = torch.topk(vec.softmax(-1), 8)
        recorte_Q = ex.tok.decode([pt.ids[i] for i in pt.Q])
        tam = [len(ex.tokenizar(ex.render([{"role": "user", "content": prompt_etapa1(x.texto, x.dataset, "irrestrita")}],
                                           cfg.prefill_resposta))[0]) for x in questoes]
        out = {"calibracao": cal,
               "top8_apos_prefill": [(ex.tok.convert_ids_to_tokens(int(i)), round(float(p), 4))
                                     for p, i in zip(top.values, top.indices)],
               "Q_confere_com_questao": normalizar(recorte_Q).endswith(normalizar(q.texto)[-60:]),
               "n_tokens_etapa1": tam,
               "estimativa_tokens_com_contexto": [int(3.2 * t) for t in tam], "n_ctx": ex.n_ctx,
               "start_at_layer": ex.start_at_layer_ok,
               "fim_do_prompt": pt.texto[-300:]}
        for k, v in out.items():
            self.log.info(f"diagnóstico | {k}: {v}")
        if max(out["estimativa_tokens_com_contexto"]) > ex.n_ctx:
            self.log.warning("Alguma questão pode estourar n_ctx na etapa com contexto; aumente cfg.n_ctx.")
        return out

    def rodar(self, questoes: list[Questao]):
        cfg, log = self.cfg, self.log
        cal = getattr(self.ex, "calibracao", None) or {}
        if not self.ex.letra_ids or cal.get("prefill_escolhido") != cfg.prefill_resposta:
            self.calibrar(questoes)
        feitos = {r["chave"] for r in ler_jsonl(self.p_itens)}
        plano = []
        for q in questoes:
            for est in cfg.estrategias:
                for T in cfg.temperaturas:
                    n = 1 if T == 0 else (cfg.rodadas_t_positiva_cot if est == "cot" else cfg.rodadas_t_positiva)
                    plano += [(q, est, T, r) for r in range(n)]
        pend = [p for p in plano if chave_item(p[0].qid, p[1], p[2], p[3]) not in feitos]
        log.info(f"Plano: {len(plano)} itens, {len(plano)-len(pend)} já feitos, {len(pend)} pendentes | dir={self.run_dir}")
        t_ini = time.time()
        for k, (q, est, T, rod) in enumerate(pend, 1):
            chave = chave_item(q.qid, est, T, rod)
            t0 = time.time()
            try:
                reg = self._processar_seguro(q, est, T, rod)
                anexar_jsonl(self.p_itens, reg)
                m = reg.get("mecanistica") or {}
                log.info(f"[{k}/{len(pend)}] {chave} | ŷ={reg.get('y_hat')} alvo={reg.get('y_ce')} "
                         f"gen={reg.get('sce_utilizavel')} val={reg.get('val')} valC={reg.get('val_c')} "
                         f"ED={_fmt(reg.get('ed_char'))} CEA*={_fmt(m.get('cea_estrela'))} "
                         f"CMR={_fmt(m.get('cmr_total'))} | {time.time()-t0:.0f}s | "
                         f"faltam ~{(time.time()-t_ini)/k*(len(pend)-k)/3600:.1f} h")
            except Exception as e:
                tb = traceback.format_exc()
                log.error(f"[{k}/{len(pend)}] {chave} FALHOU: {type(e).__name__}: {e}")
                anexar_jsonl(self.p_erros, {"chave": chave, "erro": f"{type(e).__name__}: {e}", "traceback": tb,
                                            "quando": datetime.datetime.now().isoformat()})
                if "out of memory" in str(e).lower():
                    log.error("OOM: reduza lote_patching ou max_palavras_questao.")
            finally:
                if torch is not None and torch.cuda.is_available():
                    torch.cuda.empty_cache()
            try:
                exportar_csvs(self.run_dir)
            except Exception as e:
                log.warning(f"exportar_csvs falhou: {e}")
        log.info(f"Concluído em {(time.time()-t_ini)/60:.1f} min. CSVs em {self.run_dir}")
        return self.run_dir

    # ---------------------------------------------------------------------
    def _processar_seguro(self, q: Questao, est: str, T: float, rod: int) -> dict:
        """Roda o item; se faltar memória na GPU, libera e tenta de novo com lotes 4x menores."""
        cuda = torch is not None and torch.cuda.is_available()
        for tentativa in range(2):
            if cuda:
                torch.cuda.reset_peak_memory_stats()
            oom = False
            try:
                reg = self.processar(q, est, T, rod)
                if cuda:
                    reg["mem_pico_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
                reg["tentativa_oom"] = tentativa
                return reg
            except RuntimeError as e:
                if "out of memory" not in str(e).lower() or tentativa == 1:
                    raise
                oom = True
            finally:
                if tentativa == 1:
                    self.cfg.orcamento_atencao_gb *= 4
            if oom:  # fora do except: o traceback (e os tensores presos nele) já foi liberado
                self.log.warning(f"{chave_item(q.qid, est, T, rod)}: sem memória na GPU; tentando de novo com lotes menores")
                gc.collect()
                if cuda:
                    torch.cuda.empty_cache()
                self.cfg.orcamento_atencao_gb /= 4

    def processar(self, q: Questao, est: str, T: float, rod: int) -> dict:
        cfg, ex = self.cfg, self.ex
        letras = "".join(q.alternativas)
        sem = semente_de(cfg.semente, q.qid, est, T, rod)
        cot = est == "cot"
        reg: dict[str, Any] = {
            "chave": chave_item(q.qid, est, T, rod), "run_id": cfg.run_id, "modelo": cfg.modelo,
            "dataset": q.dataset, "question_id": q.qid, "materia": q.meta.get("materia"),
            "edicao": q.meta.get("edicao"), "edicao_num": q.meta.get("edicao_num"), "estrategia": est,
            "temperatura": T, "rodada": rod, "semente": sem, "correta": q.correta,
            "permutacao": q.meta.get("permutacao"), "questao": q.texto, "n_palavras_questao": len(q.texto.split()),
            "inicio": datetime.datetime.now().isoformat(),
        }
        tempos = {}
        # ---------------- etapa 1 ----------------
        t0 = time.time()
        p1 = prompt_etapa1(q.texto, q.dataset, est)
        msgs1 = [{"role": "user", "content": p1}]
        if cot:
            g = ex.gerar(msgs1, cfg.max_novos_cot, T, sem)
            y_hat = extrair_letra(g["texto"], letras)
            resp1 = g["texto"]
            reg.update(resposta1_bruta=g["texto"], resposta1_truncada=g["truncada"])
        else:
            px = ex.preparar(msgs1, q.texto, cfg.prefill_resposta)
            r1 = ex.responder_letra(px, letras, T, sem)
            y_hat = r1["letra"]
            resp1 = resposta_curta(cfg.prefill_resposta, y_hat)
            reg.update(logits_letras_x=r1["logits_letras"], massa_letras_x=r1["massa_letras"],
                       top1_token_x=r1["top1_token"], top1_eh_letra_x=r1["top1_eh_letra"], n_tokens_x=len(px.ids))
        tempos["etapa1"] = time.time() - t0
        reg["y_hat"] = y_hat
        reg["resposta_valida"] = y_hat is not None
        reg["acerto"] = (y_hat == q.correta) if y_hat else None
        if y_hat is None:
            reg["tempos"] = tempos
            return reg

        # ---------------- etapa 2 ----------------
        rng = random.Random(semente_de(cfg.semente, q.qid, "alvo", rod))
        y_ce = rng.choice([l for l in letras if l != y_hat])
        alvo_txt = q.alternativas[y_ce]
        reg.update(y_ce=y_ce, y_ce_texto=alvo_txt)
        pedido = prompt_sce(y_ce, alvo_txt, est)
        msgs2 = (msgs1 + [{"role": "assistant", "content": resp1}, {"role": "user", "content": pedido}] if cot else
                 montar_conversa_sce(q.texto, q.dataset, est, y_hat, resp1, pedido, cfg.formato_etapa2))
        reg.update(pedido_sce=pedido, formato_etapa2=cfg.formato_etapa2 if not cot else "multiturno")
        t0 = time.time()
        max_novos = min(cfg.max_novos_sce, 2 * reg.get("n_tokens_x", 600) + (600 if cot else 300 if est == "racionais" else 200))
        g2 = ex.gerar(msgs2, max_novos, T, sem + 1)
        tempos["etapa2"] = time.time() - t0
        reg.update(sce_bruta=g2["texto"], sce_n_tokens=g2["n_tokens"], sce_truncada=g2["truncada"],
                   declara_alvo=declara_alvo(g2["texto"], y_ce))
        ps = parse_sce(g2["texto"], len(letras))
        reg.update(rotulo_encontrado=ps["rotulo_encontrado"], n_alts_sce=ps["n_alts_encontradas"],
                   motivo_falha=ps["motivo"])
        utilizavel = ps["ok"]
        if utilizavel:
            # alternativas da SCE com as mesmas letras da original
            if set(ps["alternativas"]) != set(letras):
                utilizavel, reg["motivo_falha"] = False, "letras_diferentes"
        if not utilizavel:
            reg["sce_utilizavel"] = False
            reg["tempos"] = tempos
            return reg
        enunc, alts, removidos = remover_vazamento(ps["enunciado"], ps["alternativas"], q.enunciado, alvo_txt,
                                                     q.alternativas)
        reg.update(vazamento=bool(removidos), vazamento_removido=removidos, sce_enunciado=enunc, sce_alternativas=alts)
        em = metricas_edicao(q, enunc, alts)
        reg.update(em)
        reg["copia"] = bool(em["identica"] or em["ed_char"] < cfg.ed_copia)
        lang, plang = self.lingua.detectar(enunc + " " + " ".join(alts.values()))
        reg.update(lingua=lang, lingua_prob=plang, confusao_lingua=(lang is not None and lang != "pt"))
        if em["identica"]:
            utilizavel, reg["motivo_falha"] = False, "identica"
        elif em["prop_palavras"] < cfg.min_prop_palavras:
            utilizavel, reg["motivo_falha"] = False, "curta_demais"
        elif not enunc:
            utilizavel, reg["motivo_falha"] = False, "enunciado_vazio_pos_vazamento"
        reg["sce_utilizavel"] = utilizavel
        if not utilizavel:
            reg["tempos"] = tempos
            return reg
        y_ce_sce, sim_alvo, remap = mapear_por_conteudo(q.alternativas, alts, y_ce)
        y_hat_sce, sim_orig, _ = mapear_por_conteudo(q.alternativas, alts, y_hat)
        reg.update(y_ce_sce=y_ce_sce, sim_alvo=sim_alvo, alvo_remapeado=remap, y_hat_sce=y_hat_sce, sim_orig=sim_orig)
        q_ce_txt = formatar_questao(enunc, alts)
        reg["questao_ce"] = q_ce_txt

        # ---------------- etapa 3 ----------------
        t0 = time.time()
        p3 = prompt_etapa1(q_ce_txt, q.dataset, est)
        msgs_sem = [{"role": "user", "content": p3}]
        hist = msgs2 + [{"role": "assistant", "content": g2["texto"]}]
        msgs_ctx = hist + [{"role": "user", "content": p3}]
        if cot:
            gs_ = ex.gerar(msgs_sem, cfg.max_novos_cot, T, sem + 2)
            gc_ = ex.gerar(msgs_ctx, cfg.max_novos_cot, T, sem + 3)
            f_sem, f_ctx = extrair_letra(gs_["texto"], letras), extrair_letra(gc_["texto"], letras)
            reg.update(resposta_sem_ctx_bruta=gs_["texto"], resposta_ctx_bruta=gc_["texto"])
        else:
            pce = ex.preparar(msgs_sem, q_ce_txt, cfg.prefill_resposta)
            pctx = ex.preparar(msgs_ctx, q_ce_txt, cfg.prefill_resposta)
            rs = ex.responder_letra(pce, letras, T, sem + 2)
            rc = ex.responder_letra(pctx, letras, T, sem + 3)
            f_sem, f_ctx = rs["letra"], rc["letra"]
            ld_x = ex.ld(r1["vec"], y_ce, y_hat)
            ld_ce = ex.ld(rs["vec"], y_ce_sce, y_hat_sce)
            ld_ctx = ex.ld(rc["vec"], y_ce_sce, y_hat_sce)
            reg.update(logits_letras_ce=rs["logits_letras"], logits_letras_ctx=rc["logits_letras"],
                       massa_letras_ce=rs["massa_letras"], massa_letras_ctx=rc["massa_letras"],
                       n_tokens_ce=len(pce.ids), n_tokens_ctx=len(pctx.ids), ld_x=ld_x, ld_ce=ld_ce, ld_ctx=ld_ctx)
        tempos["etapa3"] = time.time() - t0
        reg.update(f_xce=f_sem, f_xce_ctx=f_ctx,
                   val=(f_sem == y_ce_sce) if f_sem else False, val_c=(f_ctx == y_ce_sce) if f_ctx else False,
                   flip=(f_sem != y_hat_sce) if f_sem else None, flip_c=(f_ctx != y_hat_sce) if f_ctx else None,
                   resposta_ce_valida=f_sem is not None, resposta_ctx_valida=f_ctx is not None)

        # ---------------- etapa 4 ----------------
        if (cot or (T != 0 and not cfg.mecanistica_t_positiva) or not cfg.rodar_mecanistica
                or getattr(ex, "model", None) is None):
            reg["tempos"] = tempos
            return reg
        self._etapa4(reg, est, px, pce, pctx, hist, pedido, p3, ld_x, ld_ce, ld_ctx, alvo_txt, tempos)
        reg["tempos"] = tempos
        return reg

    # ---------------------------------------------------------------------
    def _etapa4(self, reg, est, px, pce, pctx, hist, pedido, p3, ld_x, ld_ce, ld_ctx, alvo_txt, tempos) -> None:
        """Patching, ERT, knockouts e DLA de um item; grava em reg['mecanistica'|'blocos'|'cabecas']."""
        cfg, ex = self.cfg, self.ex
        y_ce, y_hat, y_ce_sce, y_hat_sce = reg["y_ce"], reg["y_hat"], reg["y_ce_sce"], reg["y_hat_sce"]
        q_ce_txt = reg["questao_ce"]
        mec: dict[str, Any] = {"delta_ld": ld_ce - ld_x, "sinal_delta": int(np.sign(ld_ce - ld_x))}
        motivos = []
        if abs(ld_ce - ld_x) < cfg.tau:
            motivos.append("delta_ld_pequeno")
        if reg["ed_char"] > cfg.ed_max:
            motivos.append("reescrita_global")
        mec["incluido_cea"] = not motivos
        mec["motivo_exclusao_cea"] = ";".join(motivos) or None
        blocos, cabecas, arrays = [], [], {}
        t0 = time.time()
        if mec["incluido_cea"]:
            m, blocos, arrays = patching_edicao(ex, cfg, px, pce, (y_ce, y_hat), (y_ce_sce, y_hat_sce),
                                                ld_x, ld_ce, reg["chave"])
            mec.update(m)
        if abs(ld_ce - ld_x) >= cfg.tau:
            if not blocos:
                al = alinhar_tokens(px.ids, pce.ids, cfg.juntar_blocos_gap)
                blocos = [{**b, "bloco": i, "texto_x": ex.tok.decode(px.ids[b["x_ini"]:b["x_fim"]]),
                           "texto_ce": ex.tok.decode(pce.ids[b["ce_ini"]:b["ce_fim"]]),
                           "em_Q": b["t"] in set(px.Q), "e_t": float("nan")} for i, b in enumerate(al["blocos"])]
            blocos = ert_blocos(ex, cfg, px, pce, (y_ce_sce, y_hat_sce), ld_x, ld_ce, blocos)
            erts = [b["ert"] for b in blocos if "ert" in b]
            if erts:
                mec["ert_max"], mec["ert_soma"] = float(np.max(erts)), float(np.sum(erts))
                bmax_ert = max((b for b in blocos if "ert" in b), key=lambda b: b["ert"])
                mec["bloco_max_ert"] = bmax_ert["bloco"]
                com_e = [b for b in blocos if np.isfinite(b.get("e_t", np.nan))]
                if com_e:
                    mec["bloco_max_e"] = max(com_e, key=lambda b: b["e_t"])["bloco"]
                    mec["concordancia_e_ert"] = mec["bloco_max_e"] == mec["bloco_max_ert"]
        tempos["etapa4A"] = time.time() - t0
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        if cfg.rodar_cmr:
            t0 = time.time()
            # controle: histórico de OUTRA questão (a mais recente da mesma estratégia com question_id diferente)
            outro = None
            if cfg.controle_outro_historico:
                cands = self.hist_anterior.get(est, {})
                outro = next((cands[qo] for qo in reversed(list(cands)) if qo != reg["question_id"]), None)
            msgs_outro = (outro[0] + [{"role": "user", "content": p3}]) if outro else None
            mb, cabecas, arr_b = mediacao_contexto(
                ex, cfg, pctx, ex.render(hist, gerar=False), pedido, especificacao_alvo(y_ce, alvo_txt), y_ce,
                (y_ce_sce, y_hat_sce), ld_ce, ld_ctx, pce.ids, msgs_outro, q_ce_txt, outro[1] if outro else None)
            mec.update(mb)
            arrays.update(arr_b)
            tempos["etapa4B"] = time.time() - t0
            d = self.hist_anterior.setdefault(est, {})
            d.pop(reg["question_id"], None)
            d[reg["question_id"]] = (hist, y_ce)
            while len(d) > 5:
                d.pop(next(iter(d)))
        mec["taxonomia"] = classificar_taxonomia(reg["val"], mec.get("cea_alta"))
        if mec.get("incluido_cea") and mec.get("sinal_delta") == -1:
            mec["taxonomia"] = "sinal_invertido"   # a SCE empurra o modelo para longe do alvo: grupo à parte
        mec["taxonomia_ctx"] = (
            "acerto_por_atalho_contexto" if reg["val_c"] and mec.get("cmr_alvo") is not None
            and np.isfinite(mec.get("cmr_alvo", np.nan)) and mec["cmr_alvo"] > 0.5 else
            classificar_taxonomia(reg["val_c"], mec.get("cea_alta")))
        if arrays and cfg.salvar_mapas:
            caminho = os.path.join(self.run_dir, "mapas", nome_arquivo(reg["chave"]) + ".npz")
            np.savez_compressed(caminho, **arrays)
            mec["arquivo_mapa"] = os.path.relpath(caminho, self.run_dir)
        reg["mecanistica"] = mec
        reg["blocos"] = blocos
        reg["cabecas"] = cabecas

    # ---------------------------------------------------------------------
    def mecanistica_posterior(self, filtro="todas", n_invalidas: int | None = None, refazer: bool = False) -> str:
        """Etapa 4 sobre itens já gerados (lê itens.jsonl; não gera texto de novo).

        filtro: "todas" (toda SCE utilizável sem CoT), "t0", "t05", "recorte" (válidas com |ΔLD| ≥ τ e
        ED ≤ ED_max, mais `n_invalidas` inválidas sorteadas; padrão = mesmo número de válidas), ou uma função
        reg -> bool. Recalcula as diferenças de logits com o TransformerLens (fonte para a mecânica) e grava em
        itens_mecanistica.jsonl, que a exportação sobrepõe aos itens."""
        cfg, ex, log = self.cfg, self.ex, self.log
        if ex.model is None:
            raise RuntimeError("a passada mecânica precisa do TransformerLens carregado (fase != 'geracao')")
        cal = getattr(ex, "calibracao", None) or {}
        if not ex.letra_ids:
            amb = json.load(open(os.path.join(self.run_dir, "ambiente.json"), encoding="utf-8"))
            calr = amb.get("calibracao_letras") or {}
            ex.letra_ids = {k: int(v) for k, v in calr["ids"].items()}
            ex.calibracao = calr
            cfg.prefill_resposta = calr.get("prefill_escolhido") or cfg.prefill_resposta
        destino = os.path.join(self.run_dir, "itens_mecanistica.jsonl")
        feitos = set() if refazer else {r["chave"] for r in ler_jsonl(destino)}
        regs = [r for r in ler_jsonl(self.p_itens) if r.get("sce_utilizavel") and r.get("estrategia") != "cot"
                and r.get("questao_ce")]
        ja_inline = {r["chave"] for r in regs if r.get("mecanistica")} if not refazer else set()

        def entra_recorte(r):
            return (r.get("ld_ce") is not None and abs(r["ld_ce"] - r["ld_x"]) >= cfg.tau
                    and (r.get("ed_char") or 0) <= cfg.ed_max)
        if callable(filtro):
            sel = [r for r in regs if filtro(r)]
        elif filtro == "todas":
            sel = regs
        elif filtro == "t0":
            sel = [r for r in regs if float(r["temperatura"]) == 0]
        elif filtro == "t05":
            sel = [r for r in regs if float(r["temperatura"]) > 0]
        elif filtro == "recorte":
            validas = [r for r in regs if r.get("val") and entra_recorte(r)]
            invalidas = [r for r in regs if not r.get("val") and entra_recorte(r)]
            k = len(validas) if n_invalidas is None else n_invalidas
            rng = random.Random(semente_de(cfg.semente, "recorte"))
            sel = validas + rng.sample(invalidas, min(k, len(invalidas)))
        else:
            raise ValueError(filtro)
        pend = [r for r in sel if r["chave"] not in feitos and r["chave"] not in ja_inline]
        log.info(f"Mecânica posterior | filtro={filtro if not callable(filtro) else 'função'} | selecionados {len(sel)}"
                 f" | pendentes {len(pend)} | destino {destino}")
        t_ini = time.time()
        for k, r in enumerate(pend, 1):
            t0 = time.time()
            ok = False
            try:
                out = self._mecanistica_de_registro(r)
                anexar_jsonl(destino, out)
                ok = True
                m = out.get("mecanistica") or {}
                log.info(f"[mec {k}/{len(pend)}] {r['chave']} | ΔLD={_fmt(m.get('delta_ld'))} "
                         f"CEA*={_fmt(m.get('cea_estrela'))} CMR={_fmt(m.get('cmr_total'))} | {time.time()-t0:.0f}s | "
                         f"faltam ~{(time.time()-t_ini)/k*(len(pend)-k)/3600:.1f} h")
            except Exception as e:
                erro = f"{type(e).__name__}: {e}"
                tb = traceback.format_exc()
            if not ok:
                log.error(f"[mec {k}/{len(pend)}] {r['chave']} FALHOU: {erro}")
                anexar_jsonl(self.p_erros, {"chave": r["chave"], "etapa": "mecanistica_posterior", "erro": erro,
                                            "traceback": tb, "quando": datetime.datetime.now().isoformat()})
                gc.collect()
            if torch is not None and torch.cuda.is_available():
                torch.cuda.empty_cache()
        exportar_csvs(self.run_dir)
        return destino

    def _mecanistica_de_registro(self, r: dict) -> dict:
        cfg, ex = self.cfg, self.ex
        est, letras = r["estrategia"], "".join(sorted(set(re.findall(r"^\(([A-E])\) ", r["questao"], flags=re.M))))
        resp1 = resposta_curta(cfg.prefill_resposta, r["y_hat"])
        pedido = r.get("pedido_sce") or prompt_sce(r["y_ce"], r["y_ce_texto"], est)
        msgs2 = montar_conversa_sce(r["questao"], r["dataset"], est, r["y_hat"], resp1, pedido,
                                    r.get("formato_etapa2", "multiturno"))
        hist = msgs2 + [{"role": "assistant", "content": r["sce_bruta"]}]
        p3 = prompt_etapa1(r["questao_ce"], r["dataset"], est)
        px = ex.preparar([{"role": "user", "content": prompt_etapa1(r["questao"], r["dataset"], est)}],
                         r["questao"], cfg.prefill_resposta)
        pce = ex.preparar([{"role": "user", "content": p3}], r["questao_ce"], cfg.prefill_resposta)
        pctx = ex.preparar(hist + [{"role": "user", "content": p3}], r["questao_ce"], cfg.prefill_resposta)
        vx, vce, vctx = (ex.logits_ultimo(p.ids)[0] for p in (px, pce, pctx))
        ld_x = ex.ld(vx, r["y_ce"], r["y_hat"])
        ld_ce = ex.ld(vce, r["y_ce_sce"], r["y_hat_sce"])
        ld_ctx = ex.ld(vctx, r["y_ce_sce"], r["y_hat_sce"])
        reg = dict(r)
        tempos = {}
        self._etapa4(reg, est, px, pce, pctx, hist, pedido, p3, ld_x, ld_ce, ld_ctx, r["y_ce_texto"], tempos)
        lf = lambda v: {l: float(v[ex.letra_ids[l]]) for l in letras}
        # concordância com a etapa 1–3 original (validação TL × HF na fase em duas etapas)
        concord = {"ld_x_tl": ld_x, "ld_ce_tl": ld_ce, "ld_ctx_tl": ld_ctx,
                   "dif_ld_x": ld_x - r["ld_x"] if r.get("ld_x") is not None else None,
                   "dif_ld_ce": ld_ce - r["ld_ce"] if r.get("ld_ce") is not None else None,
                   "y_hat_tl": max(lf(vx), key=lf(vx).get), "val_tl": max(lf(vce), key=lf(vce).get) == r["y_ce_sce"],
                   "val_c_tl": max(lf(vctx), key=lf(vctx).get) == r["y_ce_sce"]}
        reg["mecanistica"].update(concord)
        return {"chave": r["chave"], "mecanistica": reg["mecanistica"], "blocos": reg.get("blocos", []),
                "cabecas": reg.get("cabecas", []), "tempos_mecanistica": tempos,
                "quando": datetime.datetime.now().isoformat()}


def _fmt(v):
    try:
        return "—" if v is None or not np.isfinite(v) else f"{v:.2f}"
    except Exception:
        return str(v)


# =============================================================================
# EXPORTAÇÃO DE CSVs (a partir de itens.jsonl; não precisa do modelo)
# =============================================================================

COLS_COMP = ["chave", "run_id", "modelo", "dataset", "question_id", "materia", "edicao", "edicao_num", "estrategia",
             "temperatura", "rodada", "correta", "y_hat", "resposta_valida", "acerto", "massa_letras_x",
             "top1_eh_letra_x", "y_ce", "sce_utilizavel", "motivo_falha", "rotulo_encontrado", "n_alts_sce",
             "sce_truncada", "sce_n_tokens", "declara_alvo", "copia", "identica", "confusao_lingua", "lingua", "vazamento",
             "alvo_remapeado", "sim_alvo", "y_ce_sce", "y_hat_sce", "f_xce", "f_xce_ctx", "val", "val_c", "flip",
             "flip_c", "ed_char", "ed_palavra", "frac_edicao_enunciado", "prop_palavras", "ld_x", "ld_ce", "ld_ctx",
             "massa_letras_ce", "massa_letras_ctx", "n_tokens_x", "n_tokens_ce", "n_tokens_ctx", "n_palavras_questao", "mem_pico_gb", "tentativa_oom"]
CHAVES_ID = ["chave", "run_id", "modelo", "dataset", "question_id", "estrategia", "temperatura", "rodada"]


def ler_itens(run_dir: str) -> list[dict]:
    """itens.jsonl com a mecânica posterior (itens_mecanistica.jsonl) sobreposta por chave."""
    regs = ler_jsonl(os.path.join(run_dir, "itens.jsonl"))
    extra = {m["chave"]: m for m in ler_jsonl(os.path.join(run_dir, "itens_mecanistica.jsonl"))}
    if extra:
        for r in regs:
            m = extra.get(r["chave"])
            if m:
                r["mecanistica"], r["blocos"], r["cabecas"] = m["mecanistica"], m.get("blocos", []), m.get("cabecas", [])
                r.setdefault("tempos", {}).update(m.get("tempos_mecanistica") or {})
    ctrl = {c["chave"]: c for c in ler_jsonl(os.path.join(run_dir, "controle_outro_hist.jsonl"))}
    if ctrl:
        for r in regs:
            c = ctrl.get(r["chave"])
            if c and r.get("mecanistica") is not None:
                r["mecanistica"] = {**r["mecanistica"], **{k: v for k, v in c.items() if k != "chave"},
                                    "controle_refeito": True}
    return regs


def recalcular_controle_outro_historico(run_dir: str, executor: "Executor | None" = None) -> str:
    """Refaz só o controle 'histórico de outra questão' de todos os itens com mecânica (1 forward por item).
    Necessário para runs feitos antes de 18/10, em que, com T = 0,5, o 'outro' histórico podia ser de outra
    rodada da MESMA questão. Grava controle_outro_hist.jsonl, que ler_itens sobrepõe."""
    regs = [r for r in ler_itens(run_dir) if r.get("mecanistica") and r.get("questao_ce")]
    destino = os.path.join(run_dir, "controle_outro_hist.jsonl")
    feitos = {r["chave"] for r in ler_jsonl(destino)}
    pend = [r for r in regs if r["chave"] not in feitos]
    if not pend:
        return destino
    proprio = executor is None
    ex = executor or _executor_do_run(run_dir)
    if not ex.letra_ids:
        cal = json.load(open(os.path.join(run_dir, "ambiente.json"), encoding="utf-8")).get("calibracao_letras") or {}
        ex.letra_ids = {k: int(v) for k, v in cal["ids"].items()}
        ex.cfg.prefill_resposta = cal.get("prefill_escolhido") or ex.cfg.prefill_resposta
    pf = ex.cfg.prefill_resposta
    t0 = time.time()
    for k, r in enumerate(pend, 1):
        mesmos = [o for o in regs if o["estrategia"] == r["estrategia"] and o["question_id"] != r["question_id"]]
        if not mesmos:
            continue
        idx = regs.index(r)
        antes = [o for o in mesmos if regs.index(o) < idx]
        o = antes[-1] if antes else mesmos[0]
        resp1 = resposta_curta(pf, o["y_hat"])
        ped = o.get("pedido_sce") or prompt_sce(o["y_ce"], o["y_ce_texto"], o["estrategia"])
        hist = montar_conversa_sce(o["questao"], o["dataset"], o["estrategia"], o["y_hat"], resp1, ped,
                                   o.get("formato_etapa2", "multiturno")) + [{"role": "assistant", "content": o["sce_bruta"]}]
        p3 = prompt_etapa1(r["questao_ce"], r["dataset"], r["estrategia"])
        pt = ex.preparar(hist + [{"role": "user", "content": p3}], r["questao_ce"], pf)
        ld_o = ex.ld(ex.logits_ultimo(pt.ids)[0], r["y_ce_sce"], r["y_hat_sce"])
        m = r["mecanistica"]
        ld_ce, ld_ctx = m.get("ld_ce_tl", r["ld_ce"]), m.get("ld_ctx_tl", r["ld_ctx"])
        anexar_jsonl(destino, {"chave": r["chave"], "outro_chave": o["chave"], "ld_ctx_outro_hist": ld_o,
                               "outro_hist_alvo_igual": o["y_ce"] == r["y_ce_sce"],
                               "frac_ganho_outro_hist": (ld_o - ld_ce) / (ld_ctx - ld_ce) if ld_ctx != ld_ce else None})
        if k % 50 == 0:
            ex.log.info(f"controle outro histórico [{k}/{len(pend)}] {time.time()-t0:.0f}s")
    if proprio:
        del ex
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
    return destino


def exportar_csvs(run_dir: str) -> dict:
    regs = ler_itens(run_dir)
    out = {}
    if not regs:
        return out
    comp = []
    mec, blocos, cabecas = [], [], []
    for r in regs:
        linha = {c: r.get(c) for c in COLS_COMP}
        for etapa, s in (r.get("tempos") or {}).items():
            linha[f"seg_{etapa}"] = s
        comp.append(linha)
        ids = {c: r.get(c) for c in CHAVES_ID}
        extra = {c: r.get(c) for c in ["val", "val_c", "flip", "acerto", "ed_char", "y_hat", "y_ce", "y_ce_sce"]}
        if r.get("mecanistica"):
            mec.append({**ids, **extra, **{k: v for k, v in r["mecanistica"].items() if not isinstance(v, (dict, list))}})
        for b in r.get("blocos") or []:
            blocos.append({**ids, **{k: v for k, v in b.items()}})
        for c in r.get("cabecas") or []:
            cabecas.append({**ids, **c})
    for nome, linhas in [("comportamental", comp), ("mecanistica", mec), ("blocos", blocos), ("cabecas", cabecas)]:
        if linhas:
            df = pd.DataFrame(linhas)
            caminho = os.path.join(run_dir, f"{nome}.csv")
            df.to_csv(caminho, index=False, encoding="utf-8")
            out[nome] = caminho
    return out


# =============================================================================
# ANÁLISE (sem GPU): tabelas, intervalos, testes, taxonomia, correção
# =============================================================================


def carregar_resultados(run_dirs) -> dict[str, pd.DataFrame]:
    """Concatena os CSVs de um ou mais runs (p.ex. vários modelos). Reexporta a partir do jsonl."""
    if isinstance(run_dirs, str):
        run_dirs = [run_dirs]
    dfs: dict[str, list] = {}
    for d in run_dirs:
        exportar_csvs(d)
        for nome in ["comportamental", "mecanistica", "blocos", "cabecas"]:
            p = os.path.join(d, f"{nome}.csv")
            if os.path.exists(p):
                dfs.setdefault(nome, []).append(pd.read_csv(p))
    return {k: pd.concat(v, ignore_index=True) for k, v in dfs.items()}


def resumo_rapido(run_dir: str, n_exemplos: int = 2) -> None:
    """Checagem pós-rodada: contagens, erros, taxas principais e exemplos de SCE."""
    exportar_csvs(run_dir)
    regs = ler_itens(run_dir)
    erros_todos = ler_jsonl(os.path.join(run_dir, "erros.jsonl"))
    feitos = {r["chave"] for r in regs}
    feitos_mec = {m["chave"] for m in ler_jsonl(os.path.join(run_dir, "itens_mecanistica.jsonl"))} | \
                 {r["chave"] for r in regs if r.get("mecanistica")}
    erros, vistos = [], set()
    for e in reversed(erros_todos):   # só os ainda não resolvidos (a nova tentativa pode ter dado certo)
        ok = e["chave"] in (feitos_mec if e.get("etapa") == "mecanistica_posterior" else feitos)
        if not ok and (e["chave"], e.get("etapa")) not in vistos:
            vistos.add((e["chave"], e.get("etapa")))
            erros.append(e)
    print(f"Run: {run_dir}\nItens concluídos: {len(regs)} | erros pendentes: {len(erros)} "
          f"(registros de erro no histórico: {len(erros_todos)})")
    for e in erros[:5]:
        print(f"  ERRO {e['chave']}: {e['erro']}")
    if not regs:
        return
    d = pd.DataFrame(regs)
    print("\nMotivos de falha na geração:", d["motivo_falha"].value_counts(dropna=False).to_dict())
    cols = [c for c in ["y_hat", "correta", "y_ce", "y_ce_sce", "sce_utilizavel", "declara_alvo", "vazamento",
                        "f_xce", "f_xce_ctx",
                        "val", "val_c", "ed_char", "ld_x", "ld_ce", "ld_ctx"] if c in d]
    print(d.set_index("chave")[cols].round(2).to_string())
    if "mecanistica" in d:
        m = pd.DataFrame([{"chave": r["chave"], **{k: v for k, v in (r.get("mecanistica") or {}).items()
                                                   if not isinstance(v, (dict, list))}} for r in regs if r.get("mecanistica")])
        if len(m):
            cm = [c for c in ["incluido_cea", "motivo_exclusao_cea", "delta_ld", "n_E", "cea", "cea_estrela",
                              "cea_alta", "ert_max", "l_meia", "corr_attr_exato", "cmr_incluido", "cmr_total",
                              "cmr_alvo", "cmr_decl", "cmr_alvo_decl", "cmr_alvo_prop", "cmr_decl_prop",
                              "cmr_alvo_decl_prop", "cmr_seg_q1", "cmr_seg_pedido", "cmr_seg_sce",
                              "frac_ganho_outro_hist", "n_declaracoes", "n_cabecas_candidatas", "taxonomia",
                              "seg_patching_exato"] if c in m]
            print("\nMecanística:\n" + m.set_index("chave")[cm].round(3).to_string())
    ok = d[d.get("sce_utilizavel", False) == True]  # noqa: E712
    for _, r in ok.head(n_exemplos).iterrows():
        print("\n" + "=" * 100 + f"\n{r['chave']} | ŷ={r['y_hat']} alvo=({r['y_ce']}) | val={r['val']} val_c={r['val_c']}")
        print("--- SCE (como avaliada) ---\n" + str(r.get("questao_ce"))[:2500])
        if r.get("vazamento"):
            print("--- removido por vazamento ---", r.get("vazamento_removido"))


def _b(s):
    return s.map(lambda v: np.nan if v is None or (isinstance(v, float) and np.isnan(v)) else float(bool(v))
                 if not isinstance(v, str) else float(v.lower() == "true"))


def bootstrap_por_questao(df: pd.DataFrame, col: str, n=10000, semente=0, estatistica="media") -> tuple:
    """IC 95% por bootstrap reamostrando questões (rodadas da mesma questão ficam juntas)."""
    d = df[["question_id", col]].dropna()
    if d.empty:
        return (np.nan, np.nan, np.nan)
    v = _b(d[col]) if d[col].dtype in (object, bool) else d[col].astype(float)
    d = d.assign(v=v).dropna(subset=["v"])
    if d.empty:
        return (np.nan, np.nan, np.nan)
    por_q = d.groupby("question_id")["v"].agg(["sum", "count"])
    s, c = por_q["sum"].to_numpy(), por_q["count"].to_numpy()
    rng = np.random.default_rng(semente)
    idx = rng.integers(0, len(s), size=(n, len(s)))
    if estatistica == "mediana":
        vals = d.groupby("question_id")["v"].median().to_numpy()
        boots = np.median(vals[idx], axis=1)
        est = float(np.median(vals))
    else:
        boots = s[idx].sum(1) / c[idx].sum(1)
        est = float(s.sum() / c.sum())
    return est, float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))


def mcnemar_exato(a, b) -> dict:
    from scipy.stats import binomtest
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    n01, n10 = int((~a & b).sum()), int((a & ~b).sum())
    p = binomtest(n01, n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    return {"so_b": n01, "so_a": n10, "p": float(p)}


def permutacao_pareada(df: pd.DataFrame, col_a: str, col_b: str, n=10000, semente=0) -> dict:
    """Troca de rótulos dentro de cada questão (sign-flip das diferenças médias por questão)."""
    d = df[["question_id", col_a, col_b]].dropna().copy()
    d["dif"] = _b(d[col_b]) - _b(d[col_a])
    dif = d.groupby("question_id")["dif"].mean().dropna().to_numpy()
    obs = dif.mean() if len(dif) else np.nan
    rng = np.random.default_rng(semente)
    sinais = rng.choice([-1, 1], size=(n, len(dif)))
    nulo = (sinais * dif).mean(1)
    p = float((1 + np.sum(np.abs(nulo) >= abs(obs))) / (1 + n)) if len(dif) else np.nan
    return {"diferenca": float(obs) if len(dif) else np.nan, "p": p, "n_questoes": len(dif)}


def benjamini_hochberg(pvals) -> np.ndarray:
    p = np.asarray(pvals, float)
    n = len(p)
    ordem = np.argsort(p)
    q = p[ordem] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n)
    out[ordem] = np.clip(q, 0, 1)
    return out


def tabela_padrao(res: dict, grupos=("modelo", "dataset", "estrategia", "temperatura"), n_boot=2000) -> pd.DataFrame:
    """Acc, Gen, Val, Val_C, Flip, ED, erros de geração, CEA*, CMR — com IC por bootstrap de questões."""
    comp = res["comportamental"].copy()
    mec = res.get("mecanistica")
    linhas = []
    for chave, g in comp.groupby(list(grupos)):
        D = g[_b(g["resposta_valida"]) == 1]
        G = D[_b(D["sce_utilizavel"]) == 1]
        lin = dict(zip(grupos, chave))
        lin["n_itens"] = len(g)
        lin["n_questoes"] = g["question_id"].nunique()

        def taxa(df, col, nome):
            e, lo, hi = bootstrap_por_questao(df.assign(**{col: _b(df[col])}), col, n_boot)
            lin[nome], lin[nome + "_ic"] = 100 * e, f"[{100*lo:.1f}, {100*hi:.1f}]"

        taxa(D, "acerto", "Acc")
        taxa(D.assign(gen=_b(D["sce_utilizavel"])), "gen", "Gen")
        tent = D[_b(D["rotulo_encontrado"]).notna()]
        for col, nome in [("copia", "Copia"), ("confusao_lingua", "Lingua"), ("vazamento", "Vazamento"),
                          ("declara_alvo", "DeclaraAlvo")]:
            if col in tent:
                lin[nome] = 100 * np.nanmean(_b(tent[col])) if len(tent) else np.nan
        if len(G):
            taxa(G, "val", "Val")
            taxa(G, "val_c", "ValC")
            taxa(G, "flip", "Flip")
            lin["DeltaC"] = lin["ValC"] - lin["Val"]
            V = G[_b(G["val"]) == 1]
            lin["ED"] = V["ed_char"].mean() if len(V) else np.nan
            VC = G[_b(G["val_c"]) == 1]
            lin["ED_C"] = VC["ed_char"].mean() if len(VC) else np.nan
            lin["ED_todas"] = G["ed_char"].mean()
            lin["frac_edicao_enunciado"] = G["frac_edicao_enunciado"].mean()
        if mec is not None and len(mec):
            m = mec.merge(g[["run_id", "chave"]], on=["run_id", "chave"])
            mi = m[_b(m["incluido_cea"]) == 1] if "incluido_cea" in m else m.iloc[0:0]
            lin["n_cea"] = len(mi)
            if len(mi) and "cea_estrela" in mi:
                lin["CEA*_media"], lin["CEA*_mediana"] = mi["cea_estrela"].mean(), mi["cea_estrela"].median()
                lin["CEA_alta_%"] = 100 * np.nanmean(_b(mi["cea_alta"]))
            if "cmr_total" in m:
                mc = m[_b(m["cmr_incluido"]) == 1]
                lin["n_cmr"] = len(mc)
                if len(mc):
                    lin["CMR_total_mediana"] = mc["cmr_total"].median()
                    lin["CMR_total_IIQ"] = f"[{mc['cmr_total'].quantile(.25):.2f}, {mc['cmr_total'].quantile(.75):.2f}]"
                    lin["CMR_alvo_mediana"] = mc["cmr_alvo"].median()
                    for c in ["cmr_decl", "cmr_alvo_decl", "cmr_alvo_prop", "cmr_decl_prop", "cmr_alvo_decl_prop",
                              "cmr_seg_q1", "cmr_seg_pedido", "cmr_seg_sce", "frac_ganho_outro_hist"]:
                        if c in mc:
                            lin[c + "_mediana"] = mc[c].median()
        linhas.append(lin)
    return pd.DataFrame(linhas)


def testes_val_vs_valc(res: dict, grupos=("modelo", "dataset", "estrategia", "temperatura")) -> pd.DataFrame:
    comp = res["comportamental"]
    out = []
    for chave, g in comp.groupby(list(grupos)):
        G = g[_b(g["sce_utilizavel"]) == 1]
        if not len(G):
            continue
        pm = permutacao_pareada(G, "val", "val_c")
        mc = mcnemar_exato(_b(G["val"]).fillna(0).astype(bool), _b(G["val_c"]).fillna(0).astype(bool))
        out.append({**dict(zip(grupos, chave)), "n": len(G), "ValC_menos_Val": pm["diferenca"],
                    "p_permutacao": pm["p"], "p_mcnemar": mc["p"], "so_valC": mc["so_b"], "so_val": mc["so_a"]})
    df = pd.DataFrame(out)
    if len(df):
        df["q_bh_permutacao"] = benjamini_hochberg(df["p_permutacao"].to_numpy())
    return df


def cea_validas_vs_invalidas(res: dict, n=10000, semente=0) -> pd.DataFrame:
    mec = res["mecanistica"]
    m = mec[_b(mec["incluido_cea"]) == 1].dropna(subset=["cea_estrela"])
    out = []
    for chave, g in m.groupby(["modelo", "dataset", "estrategia"]):
        v = g[_b(g["val"]) == 1]
        i = g[_b(g["val"]) == 0]
        lin = {"modelo": chave[0], "dataset": chave[1], "estrategia": chave[2], "n_validas": len(v),
               "n_invalidas": len(i), "CEA*_validas": v["cea_estrela"].mean(), "CEA*_invalidas": i["cea_estrela"].mean()}
        if len(v) > 1 and len(i) > 1:
            rng = np.random.default_rng(semente)
            bv = rng.choice(v["cea_estrela"].to_numpy(), (n, len(v))).mean(1)
            bi = rng.choice(i["cea_estrela"].to_numpy(), (n, len(i))).mean(1)
            d = bv - bi
            lin["diferenca"] = lin["CEA*_validas"] - lin["CEA*_invalidas"]
            lin["ic95"] = f"[{np.percentile(d, 2.5):.3f}, {np.percentile(d, 97.5):.3f}]"
        out.append(lin)
    return pd.DataFrame(out)


def teste_sinal_cmr(res: dict) -> pd.DataFrame:
    from scipy.stats import binomtest
    mec = res["mecanistica"]
    m = mec[_b(mec["cmr_incluido"]) == 1]
    out = []
    for chave, g in m.groupby(["modelo", "dataset", "estrategia"]):
        for col in [c for c in ["cmr_total", "cmr_alvo", "cmr_decl", "cmr_alvo_decl", "cmr_alvo_prop",
                                "cmr_decl_prop", "cmr_alvo_decl_prop", "cmr_seg_q1", "cmr_seg_pedido",
                                "cmr_seg_sce"] if c in g]:
            x = g[col].dropna()
            pos, neg = int((x > 0).sum()), int((x < 0).sum())
            out.append({"modelo": chave[0], "dataset": chave[1], "estrategia": chave[2], "metrica": col,
                        "n": len(x), "mediana": x.median(), "q1": x.quantile(.25), "q3": x.quantile(.75),
                        "p_sinal": binomtest(pos, pos + neg, .5).pvalue if pos + neg else np.nan})
    return pd.DataFrame(out)


def taxonomia(res: dict) -> pd.DataFrame:
    mec = res["mecanistica"]
    if "taxonomia" not in mec:
        return pd.DataFrame()
    return (mec.groupby(["modelo", "dataset", "estrategia"])["taxonomia"].value_counts(dropna=False)
            .unstack(fill_value=0))


def recalcular_cea(run_dir: str, agregacao="soma", n_perm=1000, usar="r_comb", camadas=None) -> pd.DataFrame:
    """Recalcula CEA a partir dos mapas salvos, sem rodar o modelo.

    agregacao: "max" (padrão do artigo) ou "soma" sobre camadas; usar: "r_comb", "r_attr" ou "r_exato";
    camadas: None (todas) ou (ini, fim) em fração da profundidade, p.ex. (0, 0.5) para só a primeira metade
             — antes de a informação da edição se espalhar para os tokens seguintes da questão."""
    mec = pd.read_csv(os.path.join(run_dir, "mecanistica.csv"))
    sufixo = f"{agregacao}_{usar}" + (f"_L{camadas[0]:.2f}-{camadas[1]:.2f}" if camadas else "")
    out = []
    for _, r in mec.dropna(subset=["arquivo_mapa"]).iterrows():
        z = np.load(os.path.join(run_dir, r["arquivo_mapa"]))
        if usar not in z:
            continue
        mapa = z[usar]
        if camadas:
            n = mapa.shape[0]
            mapa = mapa[int(round(camadas[0] * n)): max(int(round(camadas[1] * n)), int(round(camadas[0] * n)) + 1)]
        c = calcular_cea(mapa, z["Q"].tolist(), z["E"].tolist(), n_perm, 0, agregacao)
        c.pop("e_por_posicao_E", None)
        out.append({"chave": r["chave"], **{f"{k}_{sufixo}": v for k, v in c.items()}})
    return pd.DataFrame(out)


def plotar_mapa(run_dir: str, chave_ou_arquivo: str, janela: int = 6, ax=None):
    """Mapa de calor r(ℓ, t) nas vizinhanças dos blocos editados (colunas vermelhas = E)."""
    import matplotlib.pyplot as plt
    p = chave_ou_arquivo if chave_ou_arquivo.endswith(".npz") else os.path.join(
        run_dir, "mapas", nome_arquivo(chave_ou_arquivo) + ".npz")
    z = np.load(p)
    r, E, Q, tr = z["r_comb"], set(z["E"].tolist()), z["Q"].tolist(), int(z["t_resp"])
    cols = sorted({t for e in E for t in range(e - janela, e + 2) if t in set(Q)} | {tr})
    if ax is None:
        _, ax = plt.subplots(figsize=(max(6, len(cols) * 0.35), 6))
    im = ax.imshow(r[:, cols], aspect="auto", origin="lower", cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels([z["tokens_x"][t].replace("\n", "⏎") for t in cols], rotation=90, fontsize=8)
    for i, t in enumerate(cols):
        if t in E:
            ax.get_xticklabels()[i].set_color("red")
    ax.set_ylabel("camada")
    ax.set_title(os.path.basename(p))
    plt.colorbar(im, ax=ax, label="r(ℓ,t)")
    return ax


# ---------------- correção por especialistas -------------------------------


def exportar_anotacao(run_dir: str, n=50, semente=0, estrategia="irrestrita") -> str:
    """Planilha cega (sem a resposta do modelo à SCE), estratificada entre válidas e inválidas, T=0."""
    regs = [r for r in ler_jsonl(os.path.join(run_dir, "itens.jsonl"))
            if r.get("sce_utilizavel") and r.get("estrategia") == estrategia and float(r.get("temperatura", 0)) == 0]
    if not regs:
        raise ValueError("nenhuma SCE utilizável para anotar")
    df = pd.DataFrame(regs)
    rng = np.random.default_rng(semente)
    partes = []
    for v, g in df.groupby(df["val"].astype(bool)):
        k = min(len(g), int(round(n * len(g) / len(df))))
        partes.append(g.iloc[rng.permutation(len(g))[:k]])
    am = pd.concat(partes).sample(frac=1, random_state=semente)
    am = am.assign(id_anotacao=[f"A{i:03d}" for i in range(len(am))])
    planilha = pd.DataFrame({
        "id_anotacao": am["id_anotacao"], "questao_original": am["questao"], "questao_revisada": am["questao_ce"],
        "alternativa_alvo": am.apply(lambda r: f"({r['y_ce_sce']}) {r['sce_alternativas'][r['y_ce_sce']]}", axis=1),
        "C1_alvo_unica_correta (sim/nao/ambigua)": "", "C2_plausibilidade (1-3)": "",
        "C3_tipo_edicao (fato/negacao/numero/alternativa/global)": "", "comentario": ""})
    caminho = os.path.join(run_dir, "anotacao_cega.csv")
    planilha.to_csv(caminho, index=False, encoding="utf-8")
    am[["id_anotacao", "chave", "question_id", "modelo", "estrategia", "val", "val_c"]].to_csv(
        os.path.join(run_dir, "anotacao_chave_SIGILOSA.csv"), index=False)
    return caminho


def kappa_cohen(a, b) -> float:
    a, b = pd.Series(a).astype(str), pd.Series(b).astype(str)
    cats = sorted(set(a) | set(b))
    po = float((a == b).mean())
    pe = sum(float((a == c).mean()) * float((b == c).mean()) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")


def gwet_ac1(a, b) -> float:
    a, b = pd.Series(a).astype(str), pd.Series(b).astype(str)
    cats = sorted(set(a) | set(b))
    q = len(cats)
    po = float((a == b).mean())
    pi = [((a == c).mean() + (b == c).mean()) / 2 for c in cats]
    pe = sum(p * (1 - p) for p in pi) / (q - 1) if q > 1 else 0
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")


def cruzar_validade_correcao(run_dir: str, caminho_anotacao_consenso: str, col_c1="C1", n_boot=10000) -> dict:
    """Tabela 2×2 no nível do item, φ com bootstrap e McNemar Val − Cor."""
    chave = pd.read_csv(os.path.join(run_dir, "anotacao_chave_SIGILOSA.csv"))
    an = pd.read_csv(caminho_anotacao_consenso)
    d = chave.merge(an, on="id_anotacao")
    d["cor"] = d[col_c1].astype(str).str.lower().str.strip().eq("sim")
    d["val"] = _b(d["val"]).astype(bool)
    tab = pd.crosstab(d["val"].map({True: "válida", False: "inválida"}), d["cor"].map({True: "correta", False: "incorreta"}))

    def phi(v, c):
        v, c = v.astype(float), c.astype(float)
        return float(np.corrcoef(v, c)[0, 1]) if v.std() > 0 and c.std() > 0 else np.nan

    rng = np.random.default_rng(0)
    idx = rng.integers(0, len(d), (n_boot, len(d)))
    bs = [phi(d["val"].to_numpy()[i], d["cor"].to_numpy()[i]) for i in idx]
    return {"tabela": tab, "Cor_%": 100 * d["cor"].mean(), "Val_%": 100 * d["val"].mean(),
            "erro_autoconsistente_%": 100 * (d["val"] & ~d["cor"]).mean(),
            "ambiguas_%": 100 * d[col_c1].astype(str).str.lower().str.startswith("amb").mean(),
            "phi": phi(d["val"], d["cor"]), "phi_ic95": (np.nanpercentile(bs, 2.5), np.nanpercentile(bs, 97.5)),
            "mcnemar_val_vs_cor": mcnemar_exato(d["val"], d["cor"])}


def regressao_mista(res: dict):
    """válido ~ contexto × dataset + estratégia + modelo + ano + (1 | questão), via statsmodels (Bayes variacional)."""
    from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
    comp = res["comportamental"]
    G = comp[_b(comp["sce_utilizavel"]) == 1]
    longo = pd.concat([G.assign(valido=_b(G["val"]), contexto=0), G.assign(valido=_b(G["val_c"]), contexto=1)])
    longo = longo.dropna(subset=["valido"])
    termos = ["contexto"]
    for c in ["dataset", "estrategia", "modelo"]:
        if longo[c].nunique() > 1:
            termos.append(f"C({c})")
    if longo["dataset"].nunique() > 1:
        termos.append("contexto:C(dataset)")
    if longo["edicao_num"].notna().all() and longo["edicao_num"].nunique() > 1:
        termos.append("edicao_num")
    formula = "valido ~ " + " + ".join(termos)
    modelo = BinomialBayesMixedGLM.from_formula(formula, {"questao": "0 + C(question_id)"}, longo)
    return modelo.fit_vb()


# =============================================================================
# PERFIL LINGUÍSTICO/ESTRUTURAL DAS EDIÇÕES (sem GPU; lê itens.jsonl)
# =============================================================================

_STOP_PT = set("""a o as os um uma uns umas de do da dos das em no na nos nas por pelo pela pelos pelas para pra com sem
sob sobre entre até após ante e ou mas nem que se como quando onde porque pois já não sim mais menos muito pouco ao aos à às
ser é são era foi foram será seria sendo sido ter tem têm tinha teve havia há haver este esta estes estas esse essa esses essas
aquele aquela isso isto aquilo seu sua seus suas lhe lhes ele ela eles elas qual quais cujo cuja também ainda apenas só
assim então caso conforme segundo diante sobre deste desta desse dessa nesse nessa neste nesta pelo pela""".split())
_RE_NEG = re.compile(r"\b(não|nem|nunca|jamais|sem|nega(?:m|ndo|u|ram)?|nenhum[a]?|inexist\w*|vedad\w*|proibid\w*|"
                     r"impossibilidade|ilícit\w*|ilegal|ilegítim\w*|incab\w*|indevid\w*)\b", re.I)
_RE_NUM = re.compile(r"\d|\b(um|uma|dois|duas|três|quatro|cinco|seis|sete|oito|nove|dez|onze|doze|quinze|vinte|trinta|"
                     r"cem|mil|primeir\w*|segund\w*|terceir\w*|metade|dobro|triplo)\b|R\$|%|\b(dias?|meses|mês|anos?|horas?)\b", re.I)
_RE_LEI = re.compile(r"\b(art\.?|artigo|lei|§|inciso|alínea|código|CF|CRFB|constituição|súmula|estatuto|decreto|"
                     r"resolução|provimento|regulamento|EAOAB|CLT|CPC|CPP|CDC|ECA|CTN)\b", re.I)
_RE_COMANDO = re.compile(r"(assinale|indique|marque|identifique|é correto afirmar|está correta|está correto)", re.I)


def _palavras_conteudo(s: str) -> list[str]:
    return [w for w in re.findall(r"[\wÀ-ú]+", normalizar(s).lower()) if len(w) > 2 and w not in _STOP_PT]


def _frases(s: str) -> list[str]:
    return [f.strip() for f in re.split(r"(?<=[.!?;:])\s+|\n+", s) if f.strip()]


def perfil_edicao(reg: dict) -> dict | None:
    """Descritores de uma SCE utilizável: operações, local, eco do alvo, negação, números, referências legais."""
    if not reg.get("sce_utilizavel"):
        return None
    alts_o = {}
    m = re.split(r"\n\n(?=\(A\) )", reg["questao"], maxsplit=1)
    en_o = m[0]
    for l, t in re.findall(r"^\(([A-E])\) (.*)$", m[1] if len(m) > 1 else "", flags=re.M):
        alts_o[l] = t
    en_c, alts_c = reg["sce_enunciado"], reg["sce_alternativas"]
    wo, wc = re.findall(r"\S+", normalizar(en_o)), re.findall(r"\S+", normalizar(en_c))
    sm = difflib.SequenceMatcher(None, wo, wc, autojunk=False)
    ins, dele, rep = 0, 0, 0
    inseridas, removidas, blocos = [], [], 0
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue
        blocos += 1
        if tag == "insert":
            ins += 1
        elif tag == "delete":
            dele += 1
        else:
            rep += 1
        inseridas += wc[j1:j2]
        removidas += wo[i1:i2]
    txt_ins, txt_rem = " ".join(inseridas), " ".join(removidas)
    alvo_l = reg.get("y_ce_sce") or reg["y_ce"]
    alvo_txt = alts_o.get(reg["y_ce"], reg.get("y_ce_texto", ""))
    orig_l = reg.get("y_hat")
    cont_ins = set(_palavras_conteudo(txt_ins))
    voc_en = set(_palavras_conteudo(en_o))
    novas = cont_ins - voc_en                                   # palavras de conteúdo que não estavam no enunciado
    voc_alvo = set(_palavras_conteudo(alvo_txt))
    voc_outras = set(w for l, t in alts_o.items() if l not in (reg["y_ce"],) for w in _palavras_conteudo(t))
    eco_alvo = len(novas & voc_alvo) / len(novas) if novas else float("nan")
    eco_outras = len(novas & (voc_outras - voc_alvo)) / len(novas) if novas else float("nan")
    # frases
    fo, fc = _frases(en_o), _frases(en_c)
    frases_novas = [f for f in fc if max((similaridade(f, g) for g in fo), default=0) < 0.6]
    frases_removidas = [f for f in fo if max((similaridade(f, g) for g in fc), default=0) < 0.6]
    pos_nova = None
    if frases_novas:
        idx = [fc.index(f) for f in frases_novas]
        pos_nova = "antes_do_comando" if any(i < len(fc) - 1 for i in idx) or not _RE_COMANDO.search(fc[-1]) else "no_comando"
        if any(i == len(fc) - 1 for i in idx) and _RE_COMANDO.search(fc[-1]):
            pos_nova = "no_comando"
    comando_o = next((f for f in reversed(fo) if _RE_COMANDO.search(f)), "")
    comando_c = next((f for f in reversed(fc) if _RE_COMANDO.search(f)), "")
    # alternativas
    alt_edit = {l: (similaridade(alts_o.get(l, ""), alts_c.get(l, "")) < 0.97) for l in alts_o}
    neg = bool(_RE_NEG.search(txt_ins) or _RE_NEG.search(txt_rem))
    num = bool(_RE_NUM.search(txt_ins) or _RE_NUM.search(txt_rem))
    lei = bool(_RE_LEI.search(txt_ins) or _RE_LEI.search(txt_rem))
    ed = reg.get("ed_char") or 0
    edit_en = (ins + dele + rep) > 0
    edit_alt = any(alt_edit.values())
    if ed < 2:
        tipo = "copia_ou_retoque"
    elif edit_alt and not edit_en and sorted(normalizar(t).lower() for t in alts_o.values()) == \
            sorted(normalizar(t).lower() for t in alts_c.values()):
        tipo = "permuta_alternativas"
    elif ed > 40:
        tipo = "reescrita_global"
    elif edit_alt and not edit_en:
        tipo = "so_alternativas"
    elif frases_novas and not frases_removidas and dele == 0 and rep <= 1:
        tipo = "acrescimo_de_fato"
    elif dele and not ins and not rep:
        tipo = "remocao"
    else:
        tipo = "substituicao"
    # injeção do conteúdo do alvo no enunciado: maior trecho contínuo do texto do alvo que aparece no texto inserido
    a_norm, ins_norm = normalizar(alvo_txt).lower(), normalizar(txt_ins).lower()
    if a_norm and ins_norm:
        mb = difflib.SequenceMatcher(None, a_norm, ins_norm, autojunk=False).find_longest_match(0, len(a_norm), 0, len(ins_norm))
        injecao = mb.size / len(a_norm)
    else:
        injecao = 0.0
    # permutação: as alternativas da SCE têm os mesmos conteúdos da original, em outra ordem
    conts_o = sorted(normalizar(t).lower() for t in alts_o.values())
    conts_c = sorted(normalizar(t).lower() for t in alts_c.values())
    permuta = edit_alt and conts_o == conts_c
    return {
        "injecao_alvo": injecao, "permuta_alternativas": permuta,
        "chave": reg["chave"], "run_id": reg["run_id"], "modelo": reg["modelo"], "question_id": reg["question_id"],
        "materia": reg.get("materia"), "estrategia": reg["estrategia"], "temperatura": reg["temperatura"],
        "val": reg.get("val"), "val_c": reg.get("val_c"), "acerto": reg.get("acerto"), "ed_char": ed,
        "ld_x": reg.get("ld_x"), "ld_ce": reg.get("ld_ce"), "ld_ctx": reg.get("ld_ctx"),
        "tipo": tipo, "n_blocos_palavra": blocos, "n_ins": ins, "n_rem": dele, "n_subst": rep,
        "n_palavras_inseridas": len(inseridas), "n_palavras_removidas": len(removidas),
        "delta_palavras_enunciado": len(wc) - len(wo),
        "edita_enunciado": edit_en, "edita_alternativas": edit_alt,
        "edita_alt_alvo": alt_edit.get(reg["y_ce"], False), "edita_alt_original": alt_edit.get(orig_l, False),
        "n_alts_editadas": int(sum(alt_edit.values())),
        "frases_novas": len(frases_novas), "frases_removidas": len(frases_removidas), "pos_frase_nova": pos_nova,
        "comando_alterado": similaridade(comando_o, comando_c) < 0.97 if comando_o else None,
        "negacao_envolvida": neg, "numero_prazo_valor": num, "referencia_legal": lei,
        "n_palavras_novas": len(novas), "eco_alvo": eco_alvo, "eco_outras_alts": eco_outras,
        "texto_inserido": txt_ins[:400], "texto_removido": txt_rem[:400],
        "frase_nova_exemplo": frases_novas[0][:300] if frases_novas else None,
    }


def perfil_edicoes(run_dirs) -> pd.DataFrame:
    """Uma linha por SCE utilizável, com os descritores de perfil_edicao, de um ou mais runs."""
    if isinstance(run_dirs, str):
        run_dirs = [run_dirs]
    linhas = []
    for d in run_dirs:
        for r in ler_jsonl(os.path.join(d, "itens.jsonl")):
            p = perfil_edicao(r)
            if p:
                linhas.append(p)
    return pd.DataFrame(linhas)


# =============================================================================
# CEA POR SALIÊNCIA
# -----------------------------------------------------------------------------
# Pergunta: o modelo editou o que de fato sustentava a decisão original?
#  1) Na questão ORIGINAL x, mede-se a importância de cada palavra por oclusão: os embeddings dos tokens da
#     palavra são trocados pelo embedding médio do vocabulário (o comprimento não muda) e registram-se os
#     logits das letras. Um passo por questão serve para qualquer par (ŷ, alvo): importância de w para o par
#     = M(x) − M(x \ w), com M = ℓ_ŷ − ℓ_alvo (positivo: w sustenta ŷ contra o alvo).
#  2) E = palavras de x que a SCE substituiu ou removeu (inserções ancoradas na palavra anterior).
#     CEA_sal = Σ_E imp⁺ / Σ_Q imp⁺; CEA*_sal desconta a fração de palavras editadas; teste de permutação
#     com blocos contíguos, como na CEA original.
#  3) Na estratégia racionais, o mesmo cálculo para os "trechos decisivos" declarados pelo modelo
#     (fidelidade do racional) e a sobreposição entre o que ele declarou e o que editou.
# Passo 1 precisa da GPU (calcular_saliencia); 2 e 3 rodam a partir dos arquivos (metricas_saliencia_run).
# =============================================================================

_RE_MARCADOR_ALT = re.compile(r"^\([A-E]\)$")


def palavras_com_posicao(texto: str) -> list[tuple[int, int, str]]:
    return [(m.start(), m.end(), m.group()) for m in re.finditer(r"\S+", texto)]


def palavras_editadas(questao: str, questao_ce: str) -> list[int]:
    """Índices das palavras de `questao` substituídas/removidas na SCE; inserções ancoradas na palavra anterior."""
    wx = [w for _, _, w in palavras_com_posicao(questao)]
    wc = [w for _, _, w in palavras_com_posicao(questao_ce)]
    E = set()
    for tag, i1, i2, j1, j2 in difflib.SequenceMatcher(None, wx, wc, autojunk=False).get_opcodes():
        if tag in ("replace", "delete"):
            E.update(range(i1, i2))
        elif tag == "insert":
            E.add(max(i1 - 1, 0))
    return sorted(E)


def trechos_declarados(sce_bruta: str) -> list[str]:
    """Trechos listados após 'TRECHOS DECISIVOS:' (entre aspas, ou itens de lista), antes do rótulo da SCE."""
    t = sce_bruta or ""
    m = re.search(r"TRECHOS\s+DECISIVOS\s*:?", t, re.I)
    if not m:
        return []
    fim = RE_ROTULO.search(t, m.end())
    bloco = t[m.end(): fim.start() if fim else len(t)]
    aspas = re.findall(r"[\"“”']([^\"“”']{4,}?)[\"“”']", bloco)
    if aspas:
        return [a.strip() for a in aspas]
    itens = [re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", l).strip() for l in bloco.split("\n")]
    return [i.strip("*_ ").rstrip(".;") for i in itens if len(i.strip("*_ ")) >= 4]


def palavras_dos_trechos(questao: str, trechos: list[str], limiar: float = 0.8) -> list[int]:
    """Índices das palavras de `questao` cobertas pelos trechos (busca exata normalizada, depois aproximada)."""
    pw = palavras_com_posicao(questao)
    qn = questao.lower()
    idx = set()
    for tr in trechos:
        a = normalizar(tr).lower().strip(" .;,")
        if len(a) < 4:
            continue
        ini = qn.find(a)
        if ini >= 0:
            s, e = ini, ini + len(a)
        else:
            mb = difflib.SequenceMatcher(None, a, qn, autojunk=False).find_longest_match(0, len(a), 0, len(qn))
            if mb.size < limiar * len(a):
                continue
            s, e = mb.b, mb.b + mb.size
        idx.update(i for i, (ps, pe, _) in enumerate(pw) if ps < e and pe > s)
    return sorted(idx)


def cea_de_importancias(imp: np.ndarray, Q: list[int], E: list[int], n_perm=1000, semente=0) -> dict:
    """CEA, CEA*, teste de permutação (blocos contíguos), recall no top-|E| e percentil médio de E."""
    Q = sorted(Q)
    pos = {w: i for i, w in enumerate(Q)}
    Ei = sorted(pos[w] for w in set(E) if w in pos)
    out = {"n_Q_pal": len(Q), "n_E_pal": len(Ei)}
    e = np.clip(np.asarray(imp, float)[Q], 0, None)
    tot = e.sum()
    out["imp_total_pos"] = float(tot)
    out["imp_frac_negativa"] = float(np.clip(-np.asarray(imp, float)[Q], 0, None).sum() / (np.abs(np.asarray(imp, float)[Q]).sum() + 1e-12))
    if not Ei or tot <= 0:
        out.update(cea=np.nan, cea_estrela=np.nan, p_E=np.nan, pvalor=np.nan, alta=None, recall_topk=np.nan, pct_rank_E=np.nan)
        return out
    pE = len(Ei) / len(Q)
    cea = float(e[Ei].sum() / tot)
    out.update(cea=cea, p_E=pE, cea_estrela=(cea - pE) / (1 - pE) if pE < 1 else np.nan)
    ordem = np.argsort(-np.asarray(imp, float)[Q])
    topk = set(ordem[: len(Ei)].tolist())
    out["recall_topk"] = len(topk & set(Ei)) / len(Ei)
    ranks = np.empty(len(Q)); ranks[ordem] = np.arange(len(Q))
    out["pct_rank_E"] = float(1 - ranks[Ei].mean() / max(len(Q) - 1, 1))   # 1 = editou as mais importantes
    rng = np.random.default_rng(semente)
    comp = _corridas(Ei)
    nulos = []
    for _ in range(n_perm):
        p = amostrar_posicoes_nulas(len(Q), comp, rng)
        if p is None:
            break
        nulos.append(e[p].sum() / tot)
    if nulos:
        nulos = np.array(nulos)
        out["p95_nulo"] = float(np.percentile(nulos, 95))
        out["pvalor"] = float((1 + (nulos >= cea).sum()) / (1 + len(nulos)))
        out["alta"] = bool(cea > out["p95_nulo"])
    else:
        out.update(p95_nulo=np.nan, pvalor=np.nan, alta=None)
    return out


def saliencia_questao(ex: "Executor", questao_txt: str, dataset: str, letras: str) -> dict:
    """Passo de GPU: logits das letras na questão original e com cada palavra ocluída."""
    msgs = [{"role": "user", "content": prompt_etapa1(questao_txt, dataset, "irrestrita")}]
    pt = ex.preparar(msgs, questao_txt, ex.cfg.prefill_resposta)
    base = pt.q_char[0]
    if base < 0:
        raise RuntimeError("questão não localizada no prompt")
    pw = palavras_com_posicao(questao_txt)
    toks = [[i for i in pt.Q if pt.offsets[i][0] < e + base and pt.offsets[i][1] > s + base] for s, e, _ in pw]
    media = ex.model.W_E.mean(0)
    ids_l = [ex.letra_ids[l] for l in letras]
    lg0 = ex.logits_ultimo(pt.ids)[0]
    base_l = lg0[ids_l].cpu().numpy()
    oc = np.full((len(pw), len(letras)), np.nan)
    passo = ex.lote_para(len(pt.ids), 32)
    for i0 in range(0, len(pw), passo):
        bloco = [w for w in range(i0, min(i0 + passo, len(pw))) if toks[w]]
        if not bloco:
            continue

        def ocluir(emb, hook, bloco=bloco):
            for row, w in enumerate(bloco):
                emb[row, toks[w], :] = media.to(emb.dtype)
            return emb

        lg = ex.logits_ultimo(pt.ids, fwd_hooks=[("hook_embed", ocluir)], lote=len(bloco))
        oc[bloco] = lg[:, ids_l].cpu().numpy()
    return {"letras": letras, "palavras": [w for _, _, w in pw], "logits_base": base_l.tolist(),
            "logits_oclusao": oc.tolist(), "n_tokens_palavra": [len(t) for t in toks],
            "prefill": ex.cfg.prefill_resposta, "massa_letras": float(lg0.softmax(-1)[ids_l].sum())}


def _executor_do_run(run_dir: str, log=None) -> "Executor":
    """Recria o executor com o mesmo modelo, prefill e tokens de letra usados no run."""
    cd = json.load(open(os.path.join(run_dir, "config.json"), encoding="utf-8"))
    campos = Config.__dataclass_fields__
    cfg = Config(**{k: (tuple(v) if isinstance(v, list) else v) for k, v in cd.items() if k in campos})
    cfg.geracao_backend = "transformer_lens"     # não gera texto: dispensa a cópia em bf16
    amb = json.load(open(os.path.join(run_dir, "ambiente.json"), encoding="utf-8"))
    cal = amb.get("calibracao_letras") or {}
    cfg.prefill_resposta = cal.get("prefill_escolhido") or cd.get("prefill_resposta", cfg.prefill_resposta)
    log = log or logging.getLogger("sce.saliencia")
    if not log.handlers:
        log.addHandler(logging.StreamHandler()); log.setLevel(logging.INFO)
    ex = Executor(cfg, log)
    ex.letra_ids = {k: int(v) for k, v in cal["ids"].items()}
    ex.calibracao = cal
    log.info(f"Saliência | {cfg.modelo} | prefill {cfg.prefill_resposta!r} | letras {ex.letra_ids}")
    return ex


def calcular_saliencia(run_dir: str, executor: "Executor | None" = None, so_com_sce: bool = True) -> str:
    """Passo de GPU (rápido): uma oclusão por palavra para cada questão do run. Retomável.
    Grava <run_dir>/saliencia_questoes.jsonl. Se `executor` não for dado, carrega o modelo e o libera no fim."""
    regs = ler_jsonl(os.path.join(run_dir, "itens.jsonl"))
    destino = os.path.join(run_dir, "saliencia_questoes.jsonl")
    feitas = {r["question_id"] for r in ler_jsonl(destino)}
    questoes = {}
    for r in regs:
        if r.get("estrategia") == "cot" or (so_com_sce and not r.get("sce_utilizavel")):
            continue
        questoes.setdefault(r["question_id"], r)
    pend = [q for q in questoes if q not in feitas]
    if not pend:
        return destino
    proprio = executor is None
    ex = executor or _executor_do_run(run_dir)
    t0 = time.time()
    for k, qid in enumerate(pend, 1):
        r = questoes[qid]
        letras = "".join(sorted(set(re.findall(r"^\(([A-E])\) ", r["questao"], flags=re.M))))
        s = saliencia_questao(ex, r["questao"], r["dataset"], letras)
        anexar_jsonl(destino, {"question_id": qid, "modelo": r["modelo"], **s})
        ex.log.info(f"saliência [{k}/{len(pend)}] {qid} | {len(s['palavras'])} palavras | {time.time()-t0:.0f}s")
    if proprio:
        del ex
        gc.collect()
        if torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
    return destino


def corrigir_importancias(imp: np.ndarray, n_tokens: list, Q: list[int], correcao: str) -> np.ndarray:
    """Remove o efeito genérico da oclusão (qualquer palavra ocluída reduz a margem de ŷ).
    "nenhuma": como medido; "mediana": subtrai a mediana das palavras da questão;
    "tokens": subtrai a mediana das palavras com o mesmo número de tokens (1, 2, 3, 4+), porque palavras
    mais longas perturbam mais a entrada."""
    imp = np.asarray(imp, float).copy()
    if correcao == "nenhuma":
        return imp
    Qa = np.array(Q)
    if correcao == "mediana":
        return imp - np.median(imp[Qa])
    if correcao == "tokens":
        nt = np.minimum(np.asarray(n_tokens), 4)
        out = imp.copy()
        for k in np.unique(nt[Qa]):
            sel = Qa[nt[Qa] == k]
            out[sel] = imp[sel] - np.median(imp[sel])
        return out
    raise ValueError(correcao)


def metricas_saliencia_run(run_dir: str, n_perm: int = 1000, correcao: str = "tokens") -> pd.DataFrame:
    """Sem GPU: CEA por saliência de cada SCE utilizável (todas as T) + fidelidade dos racionais.
    correcao: ver corrigir_importancias (padrão "tokens", fixado no piloto)."""
    sal = {s["question_id"]: s for s in ler_jsonl(os.path.join(run_dir, "saliencia_questoes.jsonl"))}
    linhas = []
    for r in ler_jsonl(os.path.join(run_dir, "itens.jsonl")):
        if not r.get("sce_utilizavel") or r["question_id"] not in sal or not r.get("questao_ce"):
            continue
        s = sal[r["question_id"]]
        L = {l: i for i, l in enumerate(s["letras"])}
        y, t = r["y_hat"], r["y_ce"]
        if y not in L or t not in L:
            continue
        b = np.array(s["logits_base"]); oc = np.array(s["logits_oclusao"])
        M0 = b[L[y]] - b[L[t]]
        imp = np.nan_to_num(M0 - (oc[:, L[y]] - oc[:, L[t]]))
        Q = [i for i, w in enumerate(s["palavras"]) if not _RE_MARCADOR_ALT.match(w)]
        imp = corrigir_importancias(imp, s.get("n_tokens_palavra") or [1] * len(imp), Q, correcao)
        E = palavras_editadas(r["questao"], r["questao_ce"])
        c = cea_de_importancias(imp, Q, E, n_perm, semente_de(r["chave"], "sal"))
        lin = {"chave": r["chave"], "run_id": r["run_id"], "modelo": r["modelo"], "question_id": r["question_id"],
               "estrategia": r["estrategia"], "temperatura": r["temperatura"], "val": r.get("val"), "val_c": r.get("val_c"),
               "ed_char": r.get("ed_char"), "dld": (r["ld_ce"] - r["ld_x"]) if r.get("ld_ce") is not None and r.get("ld_x") is not None else np.nan,
               "margem_original": M0, "massa_letras_sal": s.get("massa_letras"), "correcao_saliencia": correcao,
               **{f"sal_{k}": v for k, v in c.items()}}
        if r["estrategia"] == "racionais":
            tr = trechos_declarados(r.get("sce_bruta", ""))
            D = palavras_dos_trechos(r["questao"], tr)
            lin["n_trechos_declarados"] = len(tr)
            lin["n_D_pal"] = len(D)
            if D:
                cd = cea_de_importancias(imp, Q, D, n_perm, semente_de(r["chave"], "decl"))
                lin.update({f"decl_{k}": v for k, v in cd.items() if k in ("cea", "cea_estrela", "pvalor", "alta", "recall_topk", "pct_rank_E", "p_E")})
                sD, sE = set(D), set(E)
                lin["frac_declarado_editado"] = len(sD & sE) / len(sD)
                lin["frac_editado_declarado"] = len(sD & sE) / len(sE) if sE else np.nan
        linhas.append(lin)
    df = pd.DataFrame(linhas)
    if len(df):
        df.to_csv(os.path.join(run_dir, f"saliencia_{correcao}.csv"), index=False)
    return df


def resumo_saliencia(df: pd.DataFrame, n_boot: int = 2000) -> pd.DataFrame:
    """Tabela por modelo × estratégia × T, com IC por bootstrap de questões para CEA*_sal."""
    linhas = []
    for (m, est, T), g in df.groupby(["modelo", "estrategia", "temperatura"]):
        e, lo, hi = bootstrap_por_questao(g.dropna(subset=["sal_cea_estrela"]), "sal_cea_estrela", n_boot)
        lin = {"modelo": m.split("/")[-1], "estrategia": est, "temperatura": T, "n": len(g),
               "CEA*_sal_media": e, "CEA*_sal_ic": f"[{lo:.2f}, {hi:.2f}]", "CEA*_sal_mediana": g.sal_cea_estrela.median(),
               "alta_%": 100 * _b(g.sal_alta).mean(), "recall_topk": g.sal_recall_topk.mean(), "pct_rank_E": g.sal_pct_rank_E.mean()}
        if "decl_cea_estrela" in g and g.decl_cea_estrela.notna().any():
            lin.update({"CEA*_decl_mediana": g.decl_cea_estrela.median(), "decl_alta_%": 100 * _b(g.decl_alta).mean(),
                        "frac_declarado_editado": g.frac_declarado_editado.mean()})
        linhas.append(lin)
    return pd.DataFrame(linhas)


# =============================================================================
# COMPARAÇÕES ENTRE RUNS (sem GPU): fp32 × bf16, ablação de formato, fases HF × TL
# =============================================================================

def comparar_runs(run_a: str, run_b: str, rotulos=("A", "B")) -> dict:
    """Compara dois runs item a item (mesmas chaves): concordância de ŷ, Gen, Val, Val_C e diferenças nas métricas
    mecânicas. Use para fp32 × bf16 (mesmo modelo) e para a ablação de formato (mesmo modelo, outro formato)."""
    A = {r["chave"]: r for r in ler_itens(run_a)}
    B = {r["chave"]: r for r in ler_itens(run_b)}
    comuns = sorted(set(A) & set(B))
    linhas = []
    for k in comuns:
        a, b = A[k], B[k]
        ma, mb = a.get("mecanistica") or {}, b.get("mecanistica") or {}
        lin = {"chave": k, "estrategia": a["estrategia"], "temperatura": a["temperatura"],
               "y_hat_igual": a.get("y_hat") == b.get("y_hat"), "gen_a": a.get("sce_utilizavel"), "gen_b": b.get("sce_utilizavel"),
               "copia_a": a.get("motivo_falha") == "identica", "copia_b": b.get("motivo_falha") == "identica",
               "val_a": a.get("val"), "val_b": b.get("val"), "valc_a": a.get("val_c"), "valc_b": b.get("val_c"),
               "sce_igual": (a.get("questao_ce") == b.get("questao_ce")) if a.get("questao_ce") and b.get("questao_ce") else None,
               "dif_ld_x": (b["ld_x"] - a["ld_x"]) if a.get("ld_x") is not None and b.get("ld_x") is not None else np.nan}
        for c in ["cmr_total", "cmr_alvo_prop", "cmr_alvo_decl_prop", "cea_estrela", "ert_max", "l_meia"]:
            if c in ma and c in mb and ma[c] is not None and mb[c] is not None:
                lin[f"dif_{c}"] = mb[c] - ma[c]
        linhas.append(lin)
    d = pd.DataFrame(linhas)
    if d.empty:
        return {"n_comuns": 0}
    resumo = {"n_comuns": len(d), "y_hat_concordancia": d.y_hat_igual.mean(),
              "sce_identica_entre_runs": d.sce_igual.dropna().mean() if d.sce_igual.notna().any() else np.nan,
              f"Gen_{rotulos[0]}": _b(d.gen_a).mean(), f"Gen_{rotulos[1]}": _b(d.gen_b).mean(),
              f"copia_{rotulos[0]}": d.copia_a.mean(), f"copia_{rotulos[1]}": d.copia_b.mean(),
              f"Val_{rotulos[0]}": _b(d.val_a).mean(), f"Val_{rotulos[1]}": _b(d.val_b).mean(),
              f"ValC_{rotulos[0]}": _b(d.valc_a).mean(), f"ValC_{rotulos[1]}": _b(d.valc_b).mean(),
              "dif_ld_x_mediana_abs": d.dif_ld_x.abs().median()}
    for c in [c for c in d.columns if c.startswith("dif_") and c != "dif_ld_x"]:
        resumo[f"{c}_mediana_abs"] = d[c].abs().median()
    # McNemar para cópia e Val entre os runs (itens pareados)
    for nome, ca, cb in [("copia", "copia_a", "copia_b"), ("gen", "gen_a", "gen_b")]:
        x = d[[ca, cb]].dropna()
        if len(x):
            resumo[f"mcnemar_{nome}"] = mcnemar_exato(_b(x[ca]).astype(bool), _b(x[cb]).astype(bool))
    return {"resumo": resumo, "itens": d}


def validacao_fases(run_dir: str) -> dict:
    """Modelos em duas fases: concordância entre as etapas 1–3 (HF, bf16) e o recálculo no TransformerLens."""
    m = pd.DataFrame([{"chave": r["chave"], **(r.get("mecanistica") or {})} for r in ler_itens(run_dir) if r.get("mecanistica")])
    regs = {r["chave"]: r for r in ler_jsonl(os.path.join(run_dir, "itens.jsonl"))}
    if m.empty or "ld_x_tl" not in m:
        return {}
    m["y_hat_hf"] = m.chave.map(lambda k: regs[k].get("y_hat"))
    m["val_hf"] = m.chave.map(lambda k: regs[k].get("val"))
    m["val_c_hf"] = m.chave.map(lambda k: regs[k].get("val_c"))
    return {"n": len(m), "dif_ld_x_mediana_abs": m.dif_ld_x.abs().median(), "dif_ld_ce_mediana_abs": m.dif_ld_ce.abs().median(),
            "y_hat_igual": (m.y_hat_tl == m.y_hat_hf).mean(), "val_igual": (_b(m.val_tl) == _b(m.val_hf)).mean(),
            "val_c_igual": (_b(m.val_c_tl) == _b(m.val_c_hf)).mean()}

Writing /content/sce_ptbr.py


## 13. Rodada confirmatória: 70 questões novas da OAB (protocolo congelado)
Protocolo congelado após o piloto (ver `claude/protocolo_congelado.md` no projeto): prompts das 3 estratégias, τ = 1, ED_max = 40, prefill calibrado por modelo, filtros, injeção do alvo como vazamento, CEA por saliência com correção "tokens". As 30 do piloto contam como exploratórias e as 70 novas como confirmatórias; as análises reportam as 100 juntas e as 70 separadas.

Cada modelo tem dois runs: `piloto30_<modelo>_oab` (já existe para Llama e Qwen-7B) e `conf70_<modelo>_oab`.

**Ordem em cada sessão:** (A) trilha comportamental sem mecânica, rápida, para todos os itens; (B) passada mecânica sobre os itens salvos, com T = 0 e T = 0,5. As duas são retomáveis: se a sessão cair, rode a mesma célula de novo.

**Onde rodar:** A100 para Llama-8B, Qwen-7B, Gemma-2B, Mistral-7B, Gemma-9B e Falcon-10B (fp32). G4 para Mistral-Nemo-12B e Phi-3-Medium-14B (fp32, mesma célula). Mistral-24B e Qwen-32B: seção 13.4 (G4, bf16, duas fases).

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import gc, sys, importlib, dataclasses, torch
sys.path.insert(0, '/content')
import sce_ptbr as S
importlib.reload(S)

RESULTADOS = '/content/drive/MyDrive/resultados'
IDS_PILOTO = ["oab-113.pdf-144", "oab-115.pdf-164", "oab-137.pdf-184", "oab-137.pdf-191", "oab-150.pdf-068", "oab-177.pdf-247", "oab-180.pdf-281", "oab-180.pdf-282", "oab-188.pdf-062", "oab-198.pdf-085", "oab-217.pdf-193", "oab-218.pdf-197", "oab-240.pdf-221", "oab-243.pdf-246", "oab-266.pdf-204", "oab-271.pdf-257", "oab-289.pdf-181", "oab-29.pdf-284", "oab-293.pdf-216", "oab-321.pdf-244", "oab-321.pdf-249", "oab-33.pdf-326", "oab-340.pdf-182", "oab-342.pdf-203", "oab-39.pdf-390", "oab-44.pdf-047", "oab-70.pdf-240", "oab-74.pdf-277", "oab-83.pdf-075", "oab-94.pdf-061"]
IDS_CONF = ["oab-31.pdf-305", "oab-45.pdf-055", "oab-34.pdf-336", "oab-69.pdf-230", "oab-340.pdf-183", "oab-71.pdf-253", "oab-264.pdf-186", "oab-242.pdf-245", "oab-238.pdf-203", "oab-318.pdf-218", "oab-180.pdf-284", "oab-180.pdf-280", "oab-189.pdf-073", "oab-293.pdf-215", "oab-151.pdf-078", "oab-196.pdf-066", "oab-174.pdf-225", "oab-218.pdf-198", "oab-113.pdf-143", "oab-215.pdf-174", "oab-32.pdf-322", "oab-143.pdf-252", "oab-296.pdf-017", "oab-320.pdf-235", "oab-238.pdf-197", "oab-293.pdf-223", "oab-178.pdf-261", "oab-339.pdf-180", "oab-98.pdf-019", "oab-315.pdf-187", "oab-72.pdf-265", "oab-33.pdf-330", "oab-337.pdf-155", "oab-292.pdf-208", "oab-238.pdf-205", "oab-142.pdf-235", "oab-69.pdf-227", "oab-33.pdf-331", "oab-266.pdf-203", "oab-115.pdf-168", "oab-87.pdf-011", "oab-72.pdf-262", "oab-136.pdf-181", "oab-35.pdf-345", "oab-96.pdf-083", "oab-84.pdf-088", "oab-270.pdf-245", "oab-292.pdf-213", "oab-265.pdf-199", "oab-320.pdf-234", "oab-245.pdf-266", "oab-75.pdf-290", "oab-338.pdf-168", "oab-176.pdf-237", "oab-294.pdf-229", "oab-113.pdf-142", "oab-142.pdf-238", "oab-215.pdf-166", "oab-173.pdf-212", "oab-197.pdf-070", "oab-214.pdf-163", "oab-266.pdf-208", "oab-137.pdf-188", "oab-321.pdf-247", "oab-44.pdf-048", "oab-116.pdf-172", "oab-31.pdf-312", "oab-245.pdf-273", "oab-83.pdf-077", "oab-336.pdf-151"]

todas = S.carregar_oab(DADOS)
por_id = {q.qid: q for q in todas}
Q_PILOTO = [por_id[i] for i in IDS_PILOTO]
Q_CONF = [por_id[i] for i in IDS_CONF]
print(len(Q_PILOTO), 'do piloto |', len(Q_CONF), 'confirmatórias')

BASE = S.Config(dataset='oab', caminho_dados=DADOS, estrategias=('irrestrita', 'racionais', 'explicita'),
                temperaturas=(0.0, 0.5), rodadas_t_positiva=5, geracao_backend='hf', dtype='float32', n_ctx=4096,
                rodar_mecanistica=False,       # a mecânica roda na passada (B)
                dir_saida=RESULTADOS)

def curto(m): return m.split('/')[-1]

def rid(prefixo, modelo):
    """Nome do run. O piloto do Qwen-7B válido é o _v2 (prefill corrigido)."""
    r = f'{prefixo}_{curto(modelo)}_oab'
    return r + '_v2' if prefixo == 'piloto30' and 'Qwen2.5-7B' in modelo else r

def rodar_modelo(modelo, conjuntos, filtro_mec='todas', **extra):
    """conjuntos: lista de (prefixo do run, questões). Carrega o modelo uma vez e faz (A) e (B) em cada run."""
    ex = None
    for prefixo, qs in conjuntos:
        cfg = dataclasses.replace(BASE, modelo=modelo, run_id=rid(prefixo, modelo), **extra)
        exp = S.Experimento(cfg, executor=ex)
        ex = exp.ex
        exp.diagnostico(qs)
        exp.rodar(qs)                                   # (A) comportamental
        if filtro_mec:
            exp.mecanistica_posterior(filtro_mec)       # (B) mecânica: T = 0 e T = 0,5
            S.calcular_saliencia(exp.run_dir, executor=ex)   # (C) saliência das questões (rápido)
        S.resumo_rapido(exp.run_dir, n_exemplos=0)
    del ex, exp
    gc.collect(); torch.cuda.empty_cache()

30 do piloto | 70 confirmatórias


### 13.1 Modelos até 14B (um modelo por vez; troque `MODELO` e rode de novo)
Para Llama e Qwen-7B, o piloto já existe: a célula só acrescenta a mecânica com T = 0,5 nele e roda as 70 novas. Para os outros, roda o piloto e as 70. Na primeira carga de um modelo fora da lista oficial do TransformerLens, confira no log a linha **Validação TL × HF**: correlação acima de 0,99 e top-1 igual na maioria dos prompts.

In [ ]:
MODELO = 'google/gemma-2-2b-it'
# Outras opções: 'Qwen/Qwen2.5-7B-Instruct' (use o run piloto _v2), 'google/gemma-2-2b-it', 'google/gemma-2-9b-it',
# 'mistralai/Mistral-7B-Instruct-v0.3', 'tiiuae/Falcon3-10B-Instruct',
# 'mistralai/Mistral-Nemo-Instruct-2407', 'microsoft/Phi-3-medium-4k-instruct'

rodar_modelo(MODELO, [('piloto30', Q_PILOTO), ('conf70', Q_CONF)])

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

... (7 outputs left out)

09:33:10 | INFO | Hugging Face (bf16) carregado | 36s desde o início | usado para gerar
09:33:10 | INFO | start_at_layer: diferença máxima de logits 0.00e+00 -> usando
09:33:10 | INFO | Validação TL × HF: corr mínima 0.9998 | top-1 igual em 100% | maior diferença no top-20 0.17 logits
09:33:10 | INFO | fastText indisponível (No module named 'fasttext'); usando langdetect
09:33:11 | INFO | Calibração do run reaproveitada: prefill 'Resposta: (', letras {'A': 235280, 'B': 235305, 'C': 235288, 'D': 235299, 'E': 235291}
09:33:11 | INFO | Calibração das letras: {'variante': 'com_espaco', 'massa_media': {'sem_espaco': 3.5873721026291604e-05, 'com_espaco': 0.2606770969927311}, 'token_unico': True, 'ids': {'A': 586, 'B': 599, 'C': 585, 'D': 608, 'E': 637}, 'tokens': {'A': '▁A', 'B': '▁B', 'C': '▁C', 'D': '▁D', 'E': '▁E'}}
09:33:11 | WARNING | Massa nas letras após o prefill é baixa (0.26). Revise prefill_resposta antes da rodada completa.
09:33:11 | INFO | Calibração das letras: {'variante': 's

Run: /content/drive/MyDrive/resultados/piloto30_gemma-2-2b-it_oab
Itens concluídos: 540 | erros pendentes: 0 (registros de erro no histórico: 0)

Motivos de falha na geração: {'identica': 364, None: 169, 'curta_demais': 4, 'alternativas_nao_encontradas': 3}
                                    y_hat correta y_ce y_ce_sce  sce_utilizavel  declara_alvo vazamento f_xce f_xce_ctx    val  val_c  ed_char  ld_x  ld_ce  ld_ctx
chave                                                                                                                                                              
oab-113.pdf-144|irrestrita|T0.00|r0     D       B    B      NaN           False         False     False   NaN       NaN    NaN    NaN     0.00   NaN    NaN     NaN
oab-113.pdf-144|irrestrita|T0.50|r0     D       B    B      NaN           False         False     False   NaN       NaN    NaN    NaN     0.00   NaN    NaN     NaN
oab-113.pdf-144|irrestrita|T0.50|r1     D       B    C      NaN           False       

09:33:14 | INFO | fastText indisponível (No module named 'fasttext'); usando langdetect
09:33:15 | INFO | Calibração das letras: {'variante': 'com_espaco', 'massa_media': {'sem_espaco': 2.0008950923511292e-05, 'com_espaco': 0.17340621650218963}, 'token_unico': True, 'ids': {'A': 586, 'B': 599, 'C': 585, 'D': 608, 'E': 637}, 'tokens': {'A': '▁A', 'B': '▁B', 'C': '▁C', 'D': '▁D', 'E': '▁E'}}
09:33:15 | WARNING | Massa nas letras após o prefill é baixa (0.17). Revise prefill_resposta antes da rodada completa.
09:33:15 | INFO | Calibração das letras: {'variante': 'sem_espaco', 'massa_media': {'sem_espaco': 0.9900731086730957, 'com_espaco': 2.2475456171378026e-05}, 'token_unico': True, 'ids': {'A': 235280, 'B': 235305, 'C': 235288, 'D': 235299, 'E': 235291}, 'tokens': {'A': 'A', 'B': 'B', 'C': 'C', 'D': 'D', 'E': 'E'}}
09:33:15 | INFO | Calibração das letras: {'variante': 'sem_espaco', 'massa_media': {'sem_espaco': 0.9900731086730957, 'com_espaco': 2.2475456171378026e-05}, 'token_unico': Tr

Run: /content/drive/MyDrive/resultados/conf70_gemma-2-2b-it_oab
Itens concluídos: 1260 | erros pendentes: 0 (registros de erro no histórico: 0)

Motivos de falha na geração: {'identica': 674, None: 581, 'alternativas_nao_encontradas': 5}
                                    y_hat correta y_ce y_ce_sce  sce_utilizavel  declara_alvo vazamento f_xce f_xce_ctx    val  val_c  ed_char  ld_x  ld_ce  ld_ctx
chave                                                                                                                                                              
oab-31.pdf-305|irrestrita|T0.00|r0      D       D    C      NaN           False         False     False   NaN       NaN    NaN    NaN     0.00   NaN    NaN     NaN
oab-31.pdf-305|irrestrita|T0.50|r0      D       D    C      NaN           False         False     False   NaN       NaN    NaN    NaN     0.00   NaN    NaN     NaN
oab-31.pdf-305|irrestrita|T0.50|r1      D       D    C      NaN           False         False     False   